# AIC26 Object Detection v5 — runtime-locked Colab A100 + R2 + Elasticsearch

Pipeline production cho 382K keyframe:

`R2 keyframe → batch YOLOE-26-L (genre prompt thật) → Grounding-DINO fallback → canonical dedup/count → spatial + color metadata → immutable R2 shards → Elasticsearch nested documents`

Các thay đổi chính so với notebook cũ:

- đọc keyframe trực tiếp từ R2 bằng S3 API, tải song song và overlap I/O với GPU;
- batch mặc định 32 ảnh ở `imgsz=960` cho A100, tự chia batch nếu OOM;
- prompt YOLOE theo đúng genre trước inference, không prompt global rồi mới lọc;
- confidence lưu tách khỏi threshold count theo class/model; chỉ entity rời rạc mới được đếm;
- gộp nhãn đồng nghĩa/role về canonical entity để tránh đếm một người nhiều lần;
- lưu cả bbox pixel lẫn bbox chuẩn hóa, center, 7×7 grid và tag `top_left/...`;
- màu dùng YOLOE segmentation mask + CIE Lab cho whitelist; fallback được gắn reliability thấp;
- output shard `.jsonl.gz` chỉ commit bằng marker có checksum khi không có frame lỗi;
- mapping Elasticsearch dùng `nested`, giữ đúng label + bbox + màu + vị trí của cùng object;
- query helper có ranking hình học, count-zero semantics và atomic Elasticsearch alias release.

## Colab Secrets bắt buộc

`R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ACCOUNT_ID`, `R2_BUCKET`.

Tùy chọn: `R2_SESSION_TOKEN`, `R2_PUBLIC_BASE_URL`, `ELASTIC_URL`, `ELASTIC_API_KEY`, `ELASTIC_INDEX`, `ELASTIC_ALIAS`.

> Count từ detector là **ước lượng instance nhìn thấy**, không phải ground-truth. Occlusion/crowd có thể làm thiếu; metadata có `object_counts.is_lower_bound`, `lower_bound_reasons` và `count_uncertain_labels` theo class để query không diễn giải quá mức.


In [ ]:
# === 1. Cài dependency + kiểm tra GPU ===
%pip -q install "ultralytics==8.4.67" "transformers==5.14.1" "huggingface-hub==1.24.0" "safetensors==0.8.0" "boto3==1.43.51" "botocore==1.43.51" "requests==2.32.4" "ftfy==6.3.1" "Pillow==11.3.0" "git+https://github.com/ultralytics/CLIP.git@c4b6ea0932a2c0f39a0fa528af5ec4982ff15cab"

import importlib, os, platform, sys

# Colab A100/Python 3.12 preload native `_imaging` 11.3.0 để render notebook. Giữ Pillow 11.3.0
# và reload namespace PIL để không trộn Python files với extension native đã nằm trong process.
for module_name in tuple(sys.modules):
    if module_name == "PIL" or module_name.startswith("PIL."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()
import PIL
from PIL import Image as _PillowImportCheck
import torch

assert torch.cuda.is_available(), "Notebook này cần GPU runtime. Colab: Runtime > Change runtime type > A100 GPU."
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print("Python:", platform.python_version())
print("Pillow:", PIL.__version__)
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", gpu_name, "| VRAM: %.1f GiB" % gpu_gib)
if "A100" not in gpu_name.upper():
    print("WARNING: cấu hình batch/imgsz được tune cho A100; cơ chế OOM split vẫn giúp chạy trên GPU nhỏ hơn.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True


In [ ]:
# === 2. Secrets + cấu hình production ===
import hashlib, json, os
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name, default=None, required=False):
    value = os.environ.get(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    value = value or default
    if required and not value:
        raise RuntimeError(f"Thiếu secret {name}. Thêm trong Colab Secrets rồi chạy lại cell.")
    return value


# Không print credential. Access key/secret chỉ sống trong environment của runtime.
os.environ["AWS_ACCESS_KEY_ID"] = read_secret("R2_ACCESS_KEY_ID", required=True)
os.environ["AWS_SECRET_ACCESS_KEY"] = read_secret("R2_SECRET_ACCESS_KEY", required=True)
session_token = read_secret("R2_SESSION_TOKEN")
if session_token:
    os.environ["AWS_SESSION_TOKEN"] = session_token
os.environ["AWS_DEFAULT_REGION"] = "auto"

R2_ACCOUNT_ID = read_secret("R2_ACCOUNT_ID", required=True)
R2_BUCKET = read_secret("R2_BUCKET", required=True)
R2_ENDPOINT = read_secret("R2_ENDPOINT", f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com")
R2_PUBLIC_BASE_URL = (read_secret("R2_PUBLIC_BASE_URL", "") or "").rstrip("/")

# ---- Run mode ----
RUN_FULL = True                 # True = toàn bộ keyframe dưới R2_INPUT_PREFIXES
MAX_IMAGES = None               # chỉ đặt số khi debug; None cho full
R2_INPUT_PREFIXES = ["Keyframes/"]
R2_OUTPUT_ROOT = "Derived/ObjectDetection"

# ---- A100 throughput / quality ----
BATCH_SIZE = 32                 # A100 40/80GB; code tự chia đôi khi OOM
IMAGE_SIZE = 960                # tốt hơn 640 cho vật thể nhỏ trong keyframe 480p
SHARD_SIZE = 1024               # checkpoint bất biến ~32 batch/shard
DOWNLOAD_WORKERS = 64
KEEP_LOCAL_IMAGES = False       # xóa cache của shard sau khi upload thành công
KEEP_LOCAL_SHARDS = True

# Ngưỡng recall để lưu. Count dùng threshold riêng theo class/model ở dưới.
DETECTION_CONF = 0.20
COUNT_CONF = 0.35              # fallback nếu class/source chưa được calibrate
YOLO_IOU = 0.70                 # giữ người đứng sát nhau tốt hơn ngưỡng thấp
CANONICAL_DEDUP_IOU = 0.65
MAX_DETECTIONS = 300
MIN_BOX_AREA_NORM = 0.00004
YOLOE_WEIGHT = "yoloe-26l-seg.pt"  # một weight duy nhất; load lỗi thì fail-fast

# Grounding DINO chạy theo policy nhiều tín hiệu, không dùng mỗi top-confidence.
GDINO_MODE = "policy"           # off | empty | policy
GDINO_TRIGGER_TOP_CONF = 0.32
GDINO_BATCH_SIZE = 4

PIPELINE_VERSION = "aic26-od-v5"
LOCAL_ROOT = Path("/content/aic26_od")
LOCAL_CACHE = LOCAL_ROOT / "r2_cache"
LOCAL_OUTPUT = LOCAL_ROOT / "output"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)

BASE_VOCAB = [
    "person", "crowd", "child", "car", "motorcycle", "bicycle", "bus", "truck", "van",
    "train", "airplane", "boat", "traffic light", "road sign", "license plate",
    "microphone", "camera", "cell phone", "laptop", "television", "screen", "tv logo",
    "text on screen", "sign", "banner", "table", "chair", "backpack", "handbag",
    "bottle", "book", "document",
]

GENRE_VOCAB = {
    "news": [
        "reporter", "news anchor", "official", "police officer", "soldier", "firefighter",
        "ambulance", "fire truck", "fire", "smoke", "flood", "traffic accident", "protest",
        "flag", "podium", "money", "weapon", "building", "market", "hospital", "factory",
        "farm field", "helicopter",
    ],
    "cycling": [
        "cyclist", "helmet", "racing jersey", "peloton", "road", "race number", "finish line",
        "spectator", "mountain", "podium", "medal", "water bottle", "motorcycle escort",
    ],
    "lion_dance": [
        "lion dance costume", "dragon dance", "drum", "cymbal", "performer",
        "traditional costume", "firecracker", "stage", "temple", "festival decoration",
        "red lantern",
    ],
    "exam": [
        "student", "teacher", "classroom", "whiteboard", "blackboard", "notebook", "desk",
        "pen", "exam paper", "school uniform", "projector screen", "computer",
    ],
    "cooking": [
        "chef", "food", "dish", "plate", "bowl", "pan", "pot", "stove", "cutting board",
        "knife", "vegetable", "meat", "fish", "fruit", "spice", "ingredient", "kitchen",
        "cooking oil", "rice", "noodle",
    ],
    "culture": [
        "traditional costume", "ao dai", "temple", "pagoda", "statue", "handicraft",
        "traditional musical instrument", "village", "market", "festival", "lantern", "boat",
        "rice field", "landscape", "river", "mountain", "old house",
    ],
    "mekong": [
        "river", "boat", "sampan", "floating market", "rice field", "coconut tree",
        "fishing net", "fruit", "traditional house", "water buffalo", "bridge", "farmer",
        "fish", "palm tree", "wetland", "sunset", "riverbank", "market",
    ],
    "positive": [
        "young person", "volunteer", "student", "outdoor activity", "sport", "group of people",
        "stage", "city street", "nature", "smile",
    ],
}

GENRE_BY_BATCH = {
    "L21": "news", "L22": "news", "L23": "cycling", "L24": "lion_dance",
    "L25": "exam", "L26": "cooking", "L27": "culture", "L28": "mekong",
    "L29": "mekong", "L30": "positive",
}
GENRE_BY_BATCH.update({f"K{i:02d}": "news" for i in range(1, 21)})

# Role/subtype vẫn được giữ trong aliases, nhưng count dùng canonical entity.
CANONICAL_MAP = {
    "reporter": "person", "news anchor": "person", "official": "person",
    "police officer": "person", "soldier": "person", "firefighter": "person",
    "cyclist": "person", "performer": "person", "student": "person", "teacher": "person",
    "chef": "person", "farmer": "person", "volunteer": "person", "young person": "person",
    "child": "person", "spectator": "person",
    "motorcycle escort": "motorcycle", "exam paper": "document", "notebook": "book",
    "projector screen": "screen", "television": "screen", "text on screen": "text region",
    "road sign": "sign", "race number": "sign", "red lantern": "lantern",
    "group of people": "crowd", "peloton": "crowd",
}

LABEL_GROUPS = {
    "person": "people", "crowd": "people",
    "car": "vehicle", "motorcycle": "vehicle", "bicycle": "vehicle", "bus": "vehicle",
    "truck": "vehicle", "van": "vehicle", "train": "vehicle", "airplane": "vehicle",
    "helicopter": "vehicle", "boat": "vehicle", "ambulance": "vehicle", "fire truck": "vehicle",
    "screen": "electronics", "cell phone": "electronics", "laptop": "electronics",
    "camera": "electronics", "microphone": "electronics",
    "book": "document", "document": "document", "text region": "document", "sign": "document",
    "banner": "document", "tv logo": "document", "license plate": "document",
    "food": "food", "dish": "food", "plate": "food", "bowl": "food", "pan": "food",
    "pot": "food", "knife": "food", "vegetable": "food", "meat": "food", "fish": "food",
    "fruit": "food", "rice": "food", "noodle": "food",
}

VI_ALIASES = {
    "person": ["người", "con người"], "crowd": ["đám đông", "nhiều người"],
    "car": ["ô tô", "xe hơi"], "motorcycle": ["xe máy", "mô tô"],
    "bicycle": ["xe đạp"], "bus": ["xe buýt"], "truck": ["xe tải"],
    "boat": ["thuyền", "tàu"], "airplane": ["máy bay"], "helicopter": ["trực thăng"],
    "microphone": ["micro", "micrô"], "camera": ["máy quay", "máy ảnh"],
    "cell phone": ["điện thoại"], "screen": ["màn hình", "tivi"],
    "table": ["bàn"], "chair": ["ghế"], "bottle": ["chai"], "book": ["sách"],
    "document": ["tài liệu", "giấy tờ"], "sign": ["biển báo", "bảng hiệu"],
    "flag": ["lá cờ", "cờ"], "fire": ["lửa", "đám cháy"], "smoke": ["khói"],
    "flood": ["lũ", "ngập"], "food": ["đồ ăn", "thức ăn"], "knife": ["dao"],
    "plate": ["đĩa"], "bowl": ["bát", "tô"], "river": ["sông"],
}

# Chỉ đếm entity rời rạc. Scene/substance như smoke, fire, landscape, food không có
# semantics instance-count ổn định nên vẫn được detect nhưng counted=False.
COUNTABLE_CLASSES = [
    "person", "car", "motorcycle", "bicycle", "bus", "truck", "van", "train",
    "airplane", "helicopter", "boat", "ambulance", "fire truck", "microphone",
    "camera", "cell phone", "laptop", "screen", "table", "chair", "backpack",
    "handbag", "bottle", "book", "document", "sign", "banner", "license plate",
    "flag", "helmet", "plate", "bowl", "pan", "pot", "knife", "fruit",
]

# Chỉ các class có màu bề mặt/đồ vật hữu ích cho retrieval mới chạy color extraction.
# YOLOE mask được ưu tiên; Grounding-DINO-only dùng inner-box fallback và bị đánh dấu reliability thấp.
COLOR_CLASSES = [
    "person", "car", "motorcycle", "bicycle", "bus", "truck", "van", "train",
    "airplane", "helicopter", "boat", "ambulance", "fire truck", "helmet",
    "backpack", "handbag", "bottle", "chair", "table", "cell phone", "laptop",
    "screen", "flag", "sign", "banner", "book", "document", "plate", "bowl",
]

# Threshold theo detector và canonical class. Đây là điểm khởi đầu có chủ đích;
# cần recalibrate bằng ground truth của chính dataset (cell validation bên dưới).
COUNT_THRESHOLDS = {
    "yoloe": {
        "_default": 0.35, "person": 0.32, "car": 0.34, "motorcycle": 0.32,
        "bicycle": 0.32, "bus": 0.34, "truck": 0.36, "bottle": 0.38,
        "chair": 0.36, "book": 0.38, "document": 0.40, "sign": 0.38,
    },
    "gdino": {
        "_default": 0.42, "person": 0.40, "car": 0.42, "motorcycle": 0.42,
        "bicycle": 0.40, "bus": 0.42, "truck": 0.44, "helmet": 0.43,
        "bottle": 0.46, "document": 0.46, "sign": 0.45,
    },
}

# Anchor chỉ áp dụng genre chuyên biệt; không ép news phải có một class cụ thể.
GDINO_MIN_DETECTIONS_BY_GENRE = {
    "news": 1, "cycling": 2, "lion_dance": 2, "exam": 2, "cooking": 2,
    "culture": 1, "mekong": 1, "positive": 1,
}
GDINO_ANCHOR_CLASSES_BY_GENRE = {
    "cycling": ["bicycle", "helmet", "motorcycle"],
    "lion_dance": ["lion dance costume", "dragon dance", "performer", "drum"],
    "exam": ["student", "teacher", "desk", "document", "book"],
    "cooking": ["chef", "plate", "bowl", "pan", "pot", "knife"],
    "mekong": ["boat", "river", "fish", "fruit", "farmer"],
}

PALETTE = {
    "black": [0, 0, 0], "white": [255, 255, 255], "gray": [128, 128, 128],
    "red": [220, 20, 20], "orange": [255, 140, 0], "yellow": [240, 220, 30],
    "green": [30, 160, 60], "blue": [30, 80, 220], "navy": [20, 30, 90],
    "purple": [130, 40, 160], "pink": [240, 130, 180], "brown": [120, 70, 30],
    "beige": [225, 200, 160], "cyan": [40, 200, 210], "maroon": [110, 20, 30],
    "olive": [110, 110, 30],
}

cfg = {
    "pipeline_version": PIPELINE_VERSION,
    "r2": {
        "bucket": R2_BUCKET,
        "endpoint": R2_ENDPOINT,
        "public_base_url": R2_PUBLIC_BASE_URL,
        "input_prefixes": R2_INPUT_PREFIXES,
        "output_root": R2_OUTPUT_ROOT,
    },
    "run": {
        "full": RUN_FULL,
        "max_images": MAX_IMAGES,
        "shard_size": SHARD_SIZE,
        "download_workers": DOWNLOAD_WORKERS,
        "keep_local_images": KEEP_LOCAL_IMAGES,
        "keep_local_shards": KEEP_LOCAL_SHARDS,
    },
    "local": {
        "root": str(LOCAL_ROOT), "cache": str(LOCAL_CACHE), "output": str(LOCAL_OUTPUT),
    },
    "models": {
        "yoloe_weight": YOLOE_WEIGHT,
        "yoloe_text_encoder": "mobileclip2_b.ts",
        "clip_revision": "c4b6ea0932a2c0f39a0fa528af5ec4982ff15cab",
        "gdino_model": "IDEA-Research/grounding-dino-base",
        "gdino_revision": "12bdfa3120f3e7ec7b434d90674b3396eccf88eb",
        "gdino_weight_sha256": "5548f844c928c4b6f411fa8cbcc2bfa8dbbba437cb1d513975519f93c2a9ed21",
    },
    "inference": {
        "batch_size": BATCH_SIZE, "image_size": IMAGE_SIZE, "half": True,
        "detection_conf": DETECTION_CONF, "count_conf": COUNT_CONF,
        "yolo_iou": YOLO_IOU, "canonical_dedup_iou": CANONICAL_DEDUP_IOU,
        "max_detections": MAX_DETECTIONS, "min_box_area_norm": MIN_BOX_AREA_NORM,
        "gdino_mode": GDINO_MODE, "gdino_trigger_top_conf": GDINO_TRIGGER_TOP_CONF,
        "gdino_batch_size": GDINO_BATCH_SIZE, "gdino_vocab_chunk": 20, "gdino_box_threshold": 0.25,
        "gdino_text_threshold": 0.20,
        "gdino_min_detections_by_genre": GDINO_MIN_DETECTIONS_BY_GENRE,
        "gdino_anchor_classes_by_genre": GDINO_ANCHOR_CLASSES_BY_GENRE,
        "gdino_audit_rate_by_genre": {"news": 0.02, "cycling": 0.01, "lion_dance": 0.01, "exam": 0.01, "cooking": 0.01, "culture": 0.01, "mekong": 0.01, "positive": 0.01},
    },
    "metadata": {
        "grid_size": 7, "palette": PALETTE, "schema_version": "od-frame-v5",
        "countable_classes": COUNTABLE_CLASSES, "count_thresholds": COUNT_THRESHOLDS,
        "color_classes": COLOR_CLASSES, "color_space": "CIELAB",
        "color_mask_size": 64, "color_min_pixels": 80,
        # Scale mask theo chunk trên device rồi chỉ copy mask crop 64x64 về CPU.
        "color_mask_gpu_chunk": 16,
        "color_palette_version": "aic16-lab-v1",
        "color_max_delta_e_p50": 45.0, "color_min_confidence": 0.18,
        # Query màu mặc định cũng dùng min_conf=0.30; không tốn mask/Lab cho candidate yếu hơn.
        "color_min_detection_conf": 0.30,
    },
    "vocab": {"base": BASE_VOCAB, "genres": GENRE_VOCAB, "genre_by_batch": GENRE_BY_BATCH},
    "canonical_map": CANONICAL_MAP,
    "label_groups": LABEL_GROUPS,
    "vi_aliases": VI_ALIASES,
}

# Hash chỉ gồm thứ ảnh hưởng semantic/output; secret và đường dẫn local không tham gia.
hash_input = {
    "pipeline_version": cfg["pipeline_version"], "models": cfg["models"],
    "inference": cfg["inference"], "metadata": cfg["metadata"], "vocab": cfg["vocab"],
    "canonical_map": cfg["canonical_map"], "label_groups": cfg["label_groups"],
    "vi_aliases": cfg["vi_aliases"], "public_base_url": cfg["r2"]["public_base_url"],
    "input_prefixes": cfg["r2"]["input_prefixes"], "max_images": cfg["run"]["max_images"],
}
cfg_hash = hashlib.sha256(json.dumps(hash_input, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:12]
cfg["config_hash"] = cfg_hash
cfg["r2"]["output_prefix_base"] = f"{R2_OUTPUT_ROOT}/{PIPELINE_VERSION}/{cfg_hash}"
cfg["r2"]["output_prefix"] = cfg["r2"]["output_prefix_base"]

CONFIG_PATH = LOCAL_ROOT / "od_config.json"
CONFIG_PATH.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
print("Config:", CONFIG_PATH)
print("Config hash:", cfg_hash)
print("R2 input:", R2_INPUT_PREFIXES)
print("R2 output:", cfg["r2"]["output_prefix"])
print("Batch/imgsz:", BATCH_SIZE, IMAGE_SIZE, "| full:", RUN_FULL, "| max_images:", MAX_IMAGES)


In [ ]:
%%writefile od_r2.py
import gzip
import json
import os
import re
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import quote

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError


IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")
VIDEO_RE = re.compile(r"^[KL]\d{2}_V\d{3}$")


class R2Store:
    def __init__(self, cfg):
        self.cfg = cfg
        self.bucket = cfg["r2"]["bucket"]
        self.cache_root = Path(cfg["local"]["cache"]).resolve()
        self.cache_root.mkdir(parents=True, exist_ok=True)
        self.client = boto3.client(
            "s3",
            endpoint_url=cfg["r2"]["endpoint"],
            region_name="auto",
            aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
            aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
            aws_session_token=os.environ.get("AWS_SESSION_TOKEN"),
            config=Config(
                max_pool_connections=max(96, cfg["run"]["download_workers"] + 16),
                retries={"max_attempts": 8, "mode": "adaptive"},
                connect_timeout=20,
                read_timeout=120,
                s3={"addressing_style": "path"},
            ),
        )

    def check(self):
        self.client.head_bucket(Bucket=self.bucket)
        return True

    def list_keys(self, prefix):
        paginator = self.client.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=self.bucket, Prefix=prefix):
            for obj in page.get("Contents", []):
                yield obj

    def list_keyframes(self):
        seen = set()
        rows = []
        genre_map = self.cfg["vocab"]["genre_by_batch"]
        for prefix in self.cfg["r2"]["input_prefixes"]:
            for obj in self.list_keys(prefix):
                key = obj["Key"]
                if key in seen or not key.lower().endswith(IMAGE_EXTS):
                    continue
                parts = key.split("/")
                video_id = next((p for p in reversed(parts[:-1]) if VIDEO_RE.match(p)), None)
                if not video_id:
                    continue
                seen.add(key)
                frame_name = Path(parts[-1]).stem
                frame_index = int(frame_name) if frame_name.isdigit() else None
                batch = video_id.split("_", 1)[0]
                rows.append({
                    "r2_key": key,
                    "size": int(obj.get("Size", 0)),
                    "etag": str(obj.get("ETag", "")).strip('"'),
                    "video_id": video_id,
                    "frame_name": frame_name,
                    "frame_index": frame_index,
                    "batch": batch,
                    "genre": genre_map.get(batch, "news"),
                })
        rows.sort(key=lambda x: x["r2_key"])
        max_images = self.cfg["run"].get("max_images")
        return rows[:max_images] if max_images else rows

    def _safe_local_path(self, key):
        parts = Path(key).parts
        if any(p in ("", ".", "..") for p in parts):
            raise ValueError(f"Unsafe R2 key: {key!r}")
        path = (self.cache_root / Path(*parts)).resolve()
        if self.cache_root not in path.parents:
            raise ValueError(f"R2 key escapes cache root: {key!r}")
        return path

    def download_one(self, entry):
        path = self._safe_local_path(entry["r2_key"])
        expected = int(entry.get("size") or 0)
        if path.exists() and (not expected or path.stat().st_size == expected):
            out = dict(entry)
            out["local_path"] = str(path)
            return out
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name + f".part.{os.getpid()}.{threading.get_ident()}")
        try:
            response = self.client.get_object(Bucket=self.bucket, Key=entry["r2_key"])
            with open(tmp, "wb") as fout:
                shutil.copyfileobj(response["Body"], fout, length=1024 * 1024)
            if expected and tmp.stat().st_size != expected:
                raise IOError(f"Size mismatch {entry['r2_key']}: {tmp.stat().st_size} != {expected}")
            os.replace(tmp, path)
        finally:
            if tmp.exists():
                tmp.unlink()
        out = dict(entry)
        out["local_path"] = str(path)
        return out

    def stage_entries(self, entries):
        workers = self.cfg["run"]["download_workers"]
        results = [None] * len(entries)
        with ThreadPoolExecutor(max_workers=workers) as pool:
            future_to_i = {pool.submit(self.download_one, e): i for i, e in enumerate(entries)}
            for future in as_completed(future_to_i):
                results[future_to_i[future]] = future.result()
        return results

    def cleanup_entries(self, entries):
        if self.cfg["run"].get("keep_local_images"):
            return
        for entry in entries:
            raw = entry.get("local_path")
            if not raw:
                continue
            path = Path(raw).resolve()
            if self.cache_root in path.parents and path.is_file():
                path.unlink()

    def upload_file(self, local_path, key, content_type="application/octet-stream", content_encoding=None):
        extra = {"ContentType": content_type}
        if content_encoding:
            extra["ContentEncoding"] = content_encoding
        self.client.upload_file(str(local_path), self.bucket, key, ExtraArgs=extra)

    def download_file(self, key, local_path):
        local_path = Path(local_path)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        self.client.download_file(self.bucket, key, str(local_path))
        return local_path

    def object_exists(self, key):
        try:
            self.client.head_object(Bucket=self.bucket, Key=key)
            return True
        except ClientError as exc:
            code = str(exc.response.get("Error", {}).get("Code", ""))
            if code in {"404", "NoSuchKey", "NotFound"}:
                return False
            raise

    def public_url(self, key):
        base = self.cfg["r2"].get("public_base_url", "").rstrip("/")
        return f"{base}/{quote(key, safe='/')}" if base else ""


def write_gzip_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wt", encoding="utf-8", compresslevel=6) as fout:
        for row in rows:
            fout.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")
    return path


def iter_gzip_jsonl_stream(body):
    with gzip.GzipFile(fileobj=body, mode="rb") as gz:
        for raw in gz:
            if raw.strip():
                yield json.loads(raw)


In [ ]:
%%writefile od_pipeline_lib.py
import hashlib
import json
import math
import os
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import numpy as np
import torch
from PIL import Image


def batched(items, n):
    for i in range(0, len(items), n):
        yield items[i:i + n]


def clamp(v, lo, hi):
    return min(hi, max(lo, v))


def box_iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def weighted_box(a, wa, b, wb):
    den = max(1e-9, wa + wb)
    return [(a[i] * wa + b[i] * wb) / den for i in range(4)]


def position_metadata(box, width, height, grid_size=7):
    x1, y1, x2, y2 = box
    nx1, ny1 = clamp(x1 / width, 0, 1), clamp(y1 / height, 0, 1)
    nx2, ny2 = clamp(x2 / width, 0, 1), clamp(y2 / height, 0, 1)
    cx, cy = (nx1 + nx2) / 2, (ny1 + ny2) / 2
    bw, bh = max(0.0, nx2 - nx1), max(0.0, ny2 - ny1)
    col = min(grid_size - 1, max(0, int(cx * grid_size)))
    row = min(grid_size - 1, max(0, int(cy * grid_size)))
    horizontal = "left" if cx < 1 / 3 else ("right" if cx >= 2 / 3 else "center")
    vertical = "top" if cy < 1 / 3 else ("bottom" if cy >= 2 / 3 else "middle")
    region = "center" if horizontal == "center" and vertical == "middle" else f"{vertical}_{horizontal}"
    tags = sorted(set([horizontal, vertical, region, f"r{row}c{col}"]))
    return {
        "bbox_norm": {"x1": round(nx1, 6), "y1": round(ny1, 6), "x2": round(nx2, 6), "y2": round(ny2, 6)},
        "bbox_px": {
            "x1": int(round(clamp(x1, 0, width))), "y1": int(round(clamp(y1, 0, height))),
            "x2": int(round(clamp(x2, 0, width))), "y2": int(round(clamp(y2, 0, height))),
        },
        "center_norm": {"x": round(cx, 6), "y": round(cy, 6)},
        "size_norm": {"width": round(bw, 6), "height": round(bh, 6), "area": round(bw * bh, 8)},
        "grid7": {"row": row, "col": col, "id": f"r{row}c{col}"},
        "horizontal_zone": horizontal,
        "vertical_zone": vertical,
        "position": region,
        "position_tags": tags,
    }


class ColorExtractor:
    def __init__(self, cfg):
        meta = cfg["metadata"]
        self.cfg = cfg
        self.names = list(cfg["metadata"]["palette"])
        self.rgb = np.asarray([cfg["metadata"]["palette"][k] for k in self.names], dtype=np.float32)
        self.lab = self.rgb_to_lab(self.rgb)
        self.allowed = set(meta["color_classes"])
        self.size = int(meta.get("color_mask_size", 64))
        self.min_pixels = int(meta.get("color_min_pixels", 80))

    @staticmethod
    def rgb_to_lab(rgb):
        """Vectorized sRGB D65 -> CIE Lab; distance below is Delta-E 1976."""
        x = np.asarray(rgb, dtype=np.float32) / 255.0
        x = np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)
        matrix = np.asarray([
            [0.4124564, 0.3575761, 0.1804375],
            [0.2126729, 0.7151522, 0.0721750],
            [0.0193339, 0.1191920, 0.9503041],
        ], dtype=np.float32)
        xyz = x @ matrix.T
        xyz /= np.asarray([0.95047, 1.0, 1.08883], dtype=np.float32)
        delta = 6 / 29
        f = np.where(xyz > delta ** 3, np.cbrt(xyz), xyz / (3 * delta ** 2) + 4 / 29)
        return np.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], axis=1)

    def _pixels(self, image, box, canonical_label, mask_crop=None):
        W, H = image.size
        x1, y1, x2, y2 = [float(v) for v in box]
        x1, y1 = int(clamp(x1, 0, W - 1)), int(clamp(y1, 0, H - 1))
        x2, y2 = int(clamp(x2, x1 + 1, W)), int(clamp(y2, y1 + 1, H))
        roi = image.crop((x1, y1, x2, y2)).resize((self.size, self.size), Image.Resampling.BILINEAR)
        arr = np.asarray(roi, dtype=np.float32)
        keep = np.ones((self.size, self.size), dtype=bool)
        method = "inner_box_lab"
        coverage = 0.0
        if mask_crop is not None:
            mask = np.asarray(mask_crop, dtype=bool)
            if mask.shape != keep.shape:
                mask = np.asarray(
                    Image.fromarray(mask.astype(np.uint8) * 255).resize((self.size, self.size), Image.Resampling.NEAREST)
                ) > 127
            keep &= mask
            coverage = float(mask.mean())
            method = "segmentation_mask_lab"
        else:
            # Fallback cho GDINO-only/mask thiếu: inner box giảm nền nhưng reliability vẫn thấp.
            margin = max(2, int(self.size * 0.10))
            keep[:margin] = keep[-margin:] = False
            keep[:, :margin] = keep[:, -margin:] = False
        if canonical_label == "person":
            # Trên mask người chỉ giữ vùng thân/quần áo, giảm da mặt và nền/chân.
            keep[:int(self.size * 0.18)] = False
            keep[int(self.size * 0.72):] = False
            keep[:, :int(self.size * 0.10)] = False
            keep[:, int(self.size * 0.90):] = False
        pixels = arr[keep]
        if len(pixels) < self.min_pixels:
            fallback = np.ones((self.size, self.size), dtype=bool)
            margin = max(2, int(self.size * 0.12))
            fallback[:margin] = fallback[-margin:] = False
            fallback[:, :margin] = fallback[:, -margin:] = False
            if canonical_label == "person":
                fallback[:int(self.size * 0.18)] = False
                fallback[int(self.size * 0.72):] = False
            pixels = arr[fallback]
            method += "_insufficient_mask_fallback"
        # HSV-inspired exposure trimming: giữ black/white nếu ROI đồng nhất, chỉ bỏ highlight/shadow
        # cực trị khi ROI có dynamic range lớn.
        rgb01 = pixels / 255.0
        value = rgb01.max(axis=1)
        if len(value) >= self.min_pixels and float(np.quantile(value, 0.95) - np.quantile(value, 0.05)) > 0.35:
            lo, hi = np.quantile(value, [0.03, 0.97])
            trimmed = pixels[(value >= lo) & (value <= hi)]
            if len(trimmed) >= self.min_pixels:
                pixels = trimmed
        return pixels, method, coverage

    @staticmethod
    def _empty(method):
        return {
            "dominant_color": "", "color_names": [], "colors": [],
            "color_method": method, "color_confidence": 0.0,
            "color_purity": 0.0, "color_entropy": 0.0, "color_delta_e_p50": 0.0,
            "color_coverage": 0.0, "color_reliable": False,
        }

    def extract(self, image, box, canonical_label, mask_crop=None, detection_conf=1.0):
        if canonical_label not in self.allowed:
            return self._empty("skipped_not_color_class")
        if float(detection_conf) < float(self.cfg["metadata"].get("color_min_detection_conf", 0.30)):
            return self._empty("skipped_low_detection_confidence")
        pixels, method, coverage = self._pixels(image, box, canonical_label, mask_crop)
        lab = self.rgb_to_lab(pixels)
        distances = ((lab[:, None, :] - self.lab[None, :, :]) ** 2).sum(axis=2)
        ids = distances.argmin(axis=1)
        hist = np.bincount(ids, minlength=len(self.names)).astype(np.float64)
        hist /= max(1.0, hist.sum())
        order = np.argsort(-hist)[:3]
        colors = [{"name": self.names[i], "ratio": round(float(hist[i]), 4)} for i in order if hist[i] >= 0.04]
        if not colors:
            colors = [{"name": self.names[int(order[0])], "ratio": round(float(hist[order[0]]), 4)}]
        dominant_id = int(order[0])
        purity = float(hist[dominant_id])
        assigned = ids == dominant_id
        delta_e_p50 = float(np.median(np.sqrt(distances[assigned, dominant_id]))) if assigned.any() else 999.0
        nz = hist[hist > 0]
        entropy = float(-(nz * np.log(nz)).sum() / max(1e-9, math.log(len(self.names))))
        # Confidence gộp độ thuần màu và độ gần palette. Coverage được giữ riêng vì đây là
        # thuộc tính hình học của mask, không phải xác suất tên màu.
        palette_fit = max(0.0, 1.0 - delta_e_p50 / 60.0)
        confidence = purity * (0.5 + 0.5 * palette_fit)
        meta_cfg = self.cfg["metadata"]
        reliable = (
            method == "segmentation_mask_lab"
            and len(pixels) >= self.min_pixels
            and coverage >= 0.08
            and delta_e_p50 <= float(meta_cfg.get("color_max_delta_e_p50", 45.0))
            and confidence >= float(meta_cfg.get("color_min_confidence", 0.18))
        )
        return {
            "dominant_color": colors[0]["name"], "color_names": [c["name"] for c in colors],
            "colors": colors, "color_method": method, "color_confidence": round(confidence, 4),
            "color_purity": round(purity, 4), "color_entropy": round(entropy, 4),
            "color_delta_e_p50": round(delta_e_p50, 3),
            "color_coverage": round(coverage, 4), "color_reliable": reliable,
        }


class DetectionPipeline:
    def __init__(self, cfg, public_url_fn=None):
        self.cfg = cfg
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.public_url_fn = public_url_fn or (lambda _: "")
        self.color = ColorExtractor(cfg)
        self.current_genre = None
        self.half = bool(cfg["inference"].get("half") and torch.cuda.is_available())
        self.gdino_proc = self.gdino = None
        self.gdino_failed = False
        self.gdino_failure_streak = 0
        self._load_yoloe()
        self._install_ultralytics_mask_dtype_guard()
        self._initialize_yolo_text_encoder()
        # Predictor đổi model sang FP16 ở lần predict đầu tiên. Chạy một inference thật trước
        # bind_runtime để xác nhận cả model lẫn text prompt đã cùng dtype/device.
        self._warmup_yoloe()
        if cfg["inference"].get("gdino_mode", "off") != "off":
            self._load_gdino()  # pin/download/checksum trước khi runtime namespace được bind

    def _load_yoloe(self):
        from ultralytics import YOLOE
        weights = self.cfg["models"]["yoloe_weight"]
        try:
            self.yoloe = YOLOE(weights)
        except Exception as exc:
            raise RuntimeError(f"Không load được required detector {weights!r}; không fallback model: {exc!r}") from exc
        self.yoloe_name = weights
        candidates = [Path(weights), Path(str(getattr(self.yoloe, "ckpt_path", "")))]
        weight_path = next((p.resolve() for p in candidates if str(p) and p.is_file()), None)
        if weight_path is None:
            raise RuntimeError(f"Đã load {weights!r} nhưng không resolve được file checkpoint để checksum")
        digest = hashlib.sha256()
        with open(weight_path, "rb") as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
        self.weight_path = str(weight_path)
        self.weight_sha256 = digest.hexdigest()
        print("YOLOE:", weights, "| sha256:", self.weight_sha256[:16], "| half:", self.half)

    def _install_ultralytics_mask_dtype_guard(self):
        """Fix Ultralytics FP16 segmentation postprocess without changing detector precision.

        Ultralytics 8.4.67 forces mask prototypes to FP32 in process_mask/process_mask_native but
        leaves post-NMS mask coefficients in FP16. A real detection therefore reaches
        `Half @ Float`; an empty warm-up image does not. Cast only the small coefficient matrix to
        FP32, matching the precision already selected by Ultralytics for mask reconstruction.
        """
        from ultralytics.utils import ops

        if not getattr(ops.process_mask, "_aic26_fp32_coeff_guard", False):
            original_process_mask = ops.process_mask

            def process_mask_safe(protos, masks_in, bboxes, shape, upsample=False):
                return original_process_mask(protos, masks_in.float(), bboxes, shape, upsample)

            process_mask_safe._aic26_fp32_coeff_guard = True
            process_mask_safe.__name__ = original_process_mask.__name__
            process_mask_safe.__doc__ = original_process_mask.__doc__
            ops.process_mask = process_mask_safe

        if not getattr(ops.process_mask_native, "_aic26_fp32_coeff_guard", False):
            original_process_mask_native = ops.process_mask_native

            def process_mask_native_safe(protos, masks_in, bboxes, shape):
                return original_process_mask_native(protos, masks_in.float(), bboxes, shape)

            process_mask_native_safe._aic26_fp32_coeff_guard = True
            process_mask_native_safe.__name__ = original_process_mask_native.__name__
            process_mask_native_safe.__doc__ = original_process_mask_native.__doc__
            ops.process_mask_native = process_mask_native_safe

        # Probe đúng kernel từng bị bỏ qua khi ảnh warm-up không có detection.
        probe_dtype = torch.float16 if self.half else torch.float32
        with torch.inference_mode():
            protos = torch.zeros((4, 8, 8), device=self.device, dtype=probe_dtype)
            coefficients = torch.zeros((1, 4), device=self.device, dtype=probe_dtype)
            boxes = torch.tensor([[0.0, 0.0, 8.0, 8.0]], device=self.device, dtype=probe_dtype)
            mask = ops.process_mask(protos, coefficients, boxes, (8, 8), upsample=True)
        if mask.shape != (1, 8, 8) or mask.dtype != torch.uint8:
            raise RuntimeError(
                f"Ultralytics mask dtype guard self-test sai output: shape={tuple(mask.shape)} dtype={mask.dtype}"
            )
        self.segmentation_dtype_guard = "mask_coefficients_fp32_v1"
        print(
            "Ultralytics mask dtype guard passed:",
            f"coefficients {str(probe_dtype).replace('torch.', '')} -> matmul fp32",
            flush=True,
        )

    def _initialize_yolo_text_encoder(self):
        """Resolve và cache mọi genre prompt khi YOLOE vẫn còn FP32.

        Ultralytics chuyển detector sang FP16 khi khởi tạo predictor, trong khi MobileCLIP trả
        embedding FP32. Gọi set_classes() để đổi genre sau thời điểm đó sẽ đưa FP32 qua Linear
        FP16 (`Half != float`). Cache toàn bộ embedding trước rồi chỉ bind tensor đã cast sẽ vừa
        tránh lỗi dtype, vừa làm prompt của runtime bất biến.
        """
        try:
            import clip
        except Exception as exc:
            raise RuntimeError(
                "Thiếu Ultralytics CLIP đã pin; chạy lại cell dependency trước khi load detector"
            ) from exc
        genres = sorted(
            set(self.cfg["vocab"]["genres"])
            | set(self.cfg["vocab"].get("genre_by_batch", {}).values())
            | {"news"}
        )
        print(
            f"YOLOE text encoder: tạo/cache {len(genres)} genre vocabulary trước FP16/runtime binding...",
            flush=True,
        )
        core_model = self.yoloe.model
        if next(core_model.parameters()).dtype != torch.float32:
            raise RuntimeError("YOLOE phải ở FP32 trong lúc tạo text prompt cache")
        self.yolo_prompt_embeddings = {}
        prompt_digest = hashlib.sha256()
        try:
            for genre in genres:
                labels = self.labels_for_genre(genre)
                embedding = core_model.get_text_pe(labels, cache_clip_model=True).detach().float().cpu().contiguous()
                if embedding.ndim != 3 or embedding.shape[0] != 1 or embedding.shape[1] != len(labels):
                    raise RuntimeError(
                        f"Text embedding shape sai cho genre={genre}: {tuple(embedding.shape)} labels={len(labels)}"
                    )
                if not torch.isfinite(embedding).all():
                    raise RuntimeError(f"Text embedding chứa NaN/Inf cho genre={genre}")
                self.yolo_prompt_embeddings[genre] = embedding
                prompt_digest.update(
                    json.dumps(
                        {"genre": genre, "labels": labels, "shape": list(embedding.shape)},
                        sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                    ).encode("utf-8")
                )
                prompt_digest.update(embedding.numpy().tobytes())
        finally:
            # Không giữ thêm MobileCLIP 242 MB trong detector/GPU sau khi đã cache prompt.
            if hasattr(core_model, "clip_model"):
                delattr(core_model, "clip_model")
        self.prompt_embeddings_sha256 = prompt_digest.hexdigest()
        self._set_yolo_vocab("news", force=True)
        encoder_path = Path(self.cfg["models"]["yoloe_text_encoder"])
        if not encoder_path.is_file():
            raise RuntimeError(
                f"YOLOE đã set_classes nhưng không resolve được text encoder {str(encoder_path)!r}"
            )
        encoder_path = encoder_path.resolve()
        digest = hashlib.sha256()
        with open(encoder_path, "rb") as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
        self.text_encoder_path = str(encoder_path)
        self.text_encoder_sha256 = digest.hexdigest()

        clip_root = Path(clip.__file__).resolve().parent
        clip_digest = hashlib.sha256()
        for path in sorted(clip_root.rglob("*.py")):
            clip_digest.update(str(path.relative_to(clip_root)).encode("utf-8"))
            clip_digest.update(path.read_bytes())
        self.clip_code_sha256 = clip_digest.hexdigest()
        print(
            "YOLOE text encoder:", encoder_path.name,
            "| sha256:", self.text_encoder_sha256[:16],
            "| CLIP code:", self.clip_code_sha256[:16],
            "| prompts:", self.prompt_embeddings_sha256[:16],
            flush=True,
        )

    def labels_for_genre(self, genre):
        labels, seen = [], set()
        for label in self.cfg["vocab"]["base"] + self.cfg["vocab"]["genres"].get(genre, []):
            if label not in seen:
                labels.append(label)
                seen.add(label)
        return labels

    def canonical(self, label):
        return self.cfg["canonical_map"].get(label, label)

    def _set_yolo_vocab(self, genre, force=False):
        if self.current_genre == genre and not force:
            return
        labels = self.labels_for_genre(genre)
        if genre not in self.yolo_prompt_embeddings:
            raise RuntimeError(
                f"Genre {genre!r} không có trong prompt cache đã khóa; không tạo embedding động giữa runtime"
            )
        parameter = next(self.yoloe.model.parameters())
        target_dtype = torch.float16 if self.half else torch.float32
        embedding = self.yolo_prompt_embeddings[genre].to(device=parameter.device, dtype=target_dtype)
        # Gọi core API trực tiếp để luôn rebind embedding ngay cả khi tập names không đổi thứ tự.
        self.yoloe.model.set_classes(labels, embedding)
        if self.yoloe.predictor is not None:
            self.yoloe.predictor.model.names = self.yoloe.model.names
        self.current_genre = genre

    def _warmup_yoloe(self):
        """Khởi tạo predictor FP16 và fail trước runtime/R2 nếu inference stack không hợp lệ."""
        print("YOLOE warm-up: kiểm tra FP16 + text prompt trước runtime binding...", flush=True)
        probe = Image.new("RGB", (96, 64), (127, 127, 127))
        try:
            results = self._yolo_once([probe], "news")
        except Exception as exc:
            parameter = next(self.yoloe.model.parameters())
            pe = getattr(self.yoloe.model, "pe", None)
            pe_state = (
                f"{pe.dtype}@{pe.device}" if isinstance(pe, torch.Tensor) else type(pe).__name__
            )
            raise RuntimeError(
                "YOLOE warm-up thất bại trước runtime binding: "
                f"model={parameter.dtype}@{parameter.device}, prompt={pe_state}, half={self.half}: {exc!r}"
            ) from exc
        if len(results) != 1:
            raise RuntimeError(f"YOLOE warm-up trả {len(results)} result thay vì 1")
        parameter = next(self.yoloe.model.parameters())
        expected_dtype = torch.float16 if self.half else torch.float32
        if parameter.dtype != expected_dtype:
            raise RuntimeError(f"YOLOE precision thực {parameter.dtype} != cấu hình {expected_dtype}")
        # Lần đầu predictor setup đã đổi parameter sang GPU/FP16 nhưng `pe` là plain tensor;
        # rebind cache để prompt cũng cư trú đúng device/dtype cho mọi batch tiếp theo.
        self._set_yolo_vocab("news", force=True)
        pe = self.yoloe.model.pe
        if pe.dtype != parameter.dtype or pe.device != parameter.device:
            raise RuntimeError(
                f"YOLOE prompt chưa đồng bộ: model={parameter.dtype}@{parameter.device}, pe={pe.dtype}@{pe.device}"
            )
        self.yolo_precision = "fp16" if parameter.dtype == torch.float16 else "fp32"
        print(
            "YOLOE warm-up passed: model/prompt",
            f"{self.yolo_precision}@{parameter.device}",
            flush=True,
        )

    def _yolo_once(self, images, genre):
        self._set_yolo_vocab(genre)
        inf = self.cfg["inference"]
        return self.yoloe.predict(
            source=images,
            batch=len(images),
            imgsz=inf["image_size"],
            conf=inf["detection_conf"],
            iou=inf["yolo_iou"],
            max_det=inf["max_detections"],
            agnostic_nms=True,
            retina_masks=False,
            device=self.device,
            half=self.half,
            stream=False,
            verbose=False,
        )

    def predict_yolo_batch(self, images, genre):
        try:
            results = self._yolo_once(images, genre)
        except RuntimeError as exc:
            msg = str(exc).lower()
            if "out of memory" in msg and len(images) > 1:
                torch.cuda.empty_cache()
                mid = len(images) // 2
                print(f"CUDA OOM batch={len(images)}; retry {mid}+{len(images)-mid}")
                return self.predict_yolo_batch(images[:mid], genre) + self.predict_yolo_batch(images[mid:], genre)
            else:
                raise
        if len(results) != len(images):
            raise RuntimeError(f"YOLOE trả {len(results)} result cho {len(images)} ảnh")
        output = []
        for result in results:
            dets = []
            if result.boxes is not None and len(result.boxes):
                boxes = result.boxes.xyxy.detach().float().cpu().numpy()
                scores = result.boxes.conf.detach().float().cpu().numpy()
                classes = result.boxes.cls.detach().long().cpu().numpy()
                names = result.names
                mask_size = int(self.cfg["metadata"].get("color_mask_size", 64))
                mask_chunk = int(self.cfg["metadata"].get("color_mask_gpu_chunk", 16))
                color_min_conf = float(self.cfg["metadata"].get("color_min_detection_conf", 0.30))
                color_classes = set(self.cfg["metadata"]["color_classes"])
                labels = []
                for cls_id in classes:
                    label = names.get(int(cls_id), str(cls_id)) if isinstance(names, dict) else names[int(cls_id)]
                    labels.append(str(label).strip().lower())

                # Không gọi polygon accessor của Masks: API đó polygonize/copy mọi instance sang CPU.
                # Chỉ scale mask của class cần màu, crop + resize trên device, rồi copy
                # đúng 64x64 bool/detection về CPU.
                mask_crops = {}
                if result.masks is not None and len(result.masks.data):
                    from ultralytics.utils.ops import scale_masks
                    selected = [
                        i for i, label in enumerate(labels)
                        if (
                            i < len(result.masks.data)
                            and float(scores[i]) >= color_min_conf
                            and self.canonical(label) in color_classes
                        )
                    ]
                    orig_h, orig_w = [int(v) for v in result.orig_shape]
                    for ids in batched(selected, max(1, mask_chunk)):
                        mask_tensor = result.masks.data[ids].detach().float().unsqueeze(1)
                        scaled = scale_masks(mask_tensor, (orig_h, orig_w), mode="bilinear")[:, 0]
                        compact_ids, compact_masks = [], []
                        for local_i, det_i in enumerate(ids):
                            x1, y1, x2, y2 = [float(v) for v in boxes[det_i]]
                            ix1 = max(0, min(orig_w - 1, int(math.floor(x1))))
                            iy1 = max(0, min(orig_h - 1, int(math.floor(y1))))
                            ix2 = max(ix1 + 1, min(orig_w, int(math.ceil(x2))))
                            iy2 = max(iy1 + 1, min(orig_h, int(math.ceil(y2))))
                            crop = scaled[local_i:local_i + 1, None, iy1:iy2, ix1:ix2]
                            if crop.numel():
                                compact = torch.nn.functional.interpolate(
                                    crop, size=(mask_size, mask_size), mode="bilinear", align_corners=False
                                )[0, 0].ge(0.5)
                                compact_ids.append(det_i)
                                compact_masks.append(compact)
                        if compact_masks:
                            # Một device→host transfer mỗi chunk, không sync từng detection.
                            compact_cpu = torch.stack(compact_masks).to(device="cpu").numpy()
                            mask_crops.update(zip(compact_ids, compact_cpu))
                        del scaled, mask_tensor

                for det_i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
                    mask_crop = mask_crops.get(det_i)
                    dets.append({
                        "label": label, "box": [float(x) for x in box],
                        "conf": float(score), "sources": ["yoloe"],
                        "source_scores": {"yoloe": float(score)},
                        "mask_crop": mask_crop,
                        "mask_box": [float(x) for x in box] if mask_crop is not None else None,
                    })
            output.append(dets)
        return output

    def _load_gdino(self):
        if self.gdino is not None:
            return
        from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
        from huggingface_hub import hf_hub_download
        model_id = self.cfg["models"]["gdino_model"]
        revision = self.cfg["models"]["gdino_revision"]
        weight_path = hf_hub_download(model_id, filename="model.safetensors", revision=revision)
        digest = hashlib.sha256()
        with open(weight_path, "rb") as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
        actual_sha = digest.hexdigest()
        expected_sha = self.cfg["models"]["gdino_weight_sha256"]
        if actual_sha != expected_sha:
            raise RuntimeError(f"Grounding-DINO checksum mismatch: {actual_sha} != {expected_sha}")
        self.gdino_weight_sha256 = actual_sha
        self.gdino_revision = revision
        self.gdino_proc = AutoProcessor.from_pretrained(model_id, revision=revision)
        kwargs = {"dtype": torch.float16} if torch.cuda.is_available() else {}
        self.gdino = AutoModelForZeroShotObjectDetection.from_pretrained(
            model_id, revision=revision, use_safetensors=True, **kwargs
        ).to(self.device).eval()
        print("Grounding DINO fallback:", model_id, "@", revision[:12], "sha256:", actual_sha[:16])

    @staticmethod
    def _match_label(text, labels):
        text = str(text).strip().lower().strip(" .")
        if text in labels:
            return text
        candidates = [x for x in labels if x in text or text in x]
        return min(candidates, key=len) if candidates else None

    @torch.inference_mode()
    def predict_gdino_batch(self, images, genre):
        self._load_gdino()
        target_sizes = [(im.height, im.width) for im in images]
        all_labels = self.labels_for_genre(genre)
        inf = self.cfg["inference"]
        batch_out = [[] for _ in images]
        # Chunk prompt để không vượt text token limit của Grounding DINO.
        for labels in batched(all_labels, inf.get("gdino_vocab_chunk", 20)):
            # API mới nhận list-of-label-list; fallback string giữ tương thích transformers cũ.
            try:
                inputs = self.gdino_proc(images=images, text=[labels for _ in images], padding=True, return_tensors="pt")
            except Exception:
                prompt = ". ".join(labels) + "."
                inputs = self.gdino_proc(images=images, text=[prompt] * len(images), padding=True, return_tensors="pt")
            inputs = inputs.to(self.device)
            if next(self.gdino.parameters()).dtype == torch.float16 and "pixel_values" in inputs:
                inputs["pixel_values"] = inputs["pixel_values"].half()
            outputs = self.gdino(**inputs)
            try:
                processed = self.gdino_proc.post_process_grounded_object_detection(
                    outputs,
                    threshold=inf["gdino_box_threshold"],
                    text_threshold=inf["gdino_text_threshold"],
                    target_sizes=target_sizes,
                )
            except TypeError:
                try:
                    processed = self.gdino_proc.post_process_grounded_object_detection(
                        outputs,
                        inputs.get("input_ids"),
                        threshold=inf["gdino_box_threshold"],
                        text_threshold=inf["gdino_text_threshold"],
                        target_sizes=target_sizes,
                    )
                except TypeError:
                    processed = self.gdino_proc.post_process_grounded_object_detection(
                        outputs,
                        inputs.get("input_ids"),
                        box_threshold=inf["gdino_box_threshold"],
                        text_threshold=inf["gdino_text_threshold"],
                        target_sizes=target_sizes,
                    )
            for image_i, result in enumerate(processed):
                boxes = result["boxes"].detach().float().cpu().numpy()
                scores = result["scores"].detach().float().cpu().numpy()
                raw_labels = result.get("text_labels", result.get("labels", []))
                for box, score, raw in zip(boxes, scores, raw_labels):
                    label = self._match_label(raw, labels)
                    if label:
                        batch_out[image_i].append({
                            "label": label, "box": [float(x) for x in box],
                            "conf": float(score), "sources": ["gdino"],
                            "source_scores": {"gdino": float(score)},
                        })
        return batch_out

    def gdino_reasons(self, dets, genre, document_id=""):
        if self.gdino_failed:
            return []
        mode = self.cfg["inference"].get("gdino_mode", "off")
        if mode == "off":
            return []
        if not dets:
            return ["no_yolo_detection"]
        if mode == "empty":
            return []
        reasons = []
        inf = self.cfg["inference"]
        if max(d["conf"] for d in dets) < inf["gdino_trigger_top_conf"]:
            reasons.append("low_top_confidence")
        minimum = inf.get("gdino_min_detections_by_genre", {}).get(genre, 1)
        if len(dets) < minimum:
            reasons.append(f"low_detection_count_lt_{minimum}")
        anchors = set(inf.get("gdino_anchor_classes_by_genre", {}).get(genre, []))
        found = {d["label"] for d in dets} | {self.canonical(d["label"]) for d in dets}
        if anchors and not (anchors & found):
            reasons.append("missing_genre_anchor_class")
        audit_rate = float(inf.get("gdino_audit_rate_by_genre", {}).get(genre, 0.0))
        if audit_rate > 0 and document_id:
            bucket = int(hashlib.sha256(document_id.encode("utf-8")).hexdigest()[:8], 16) / 0xFFFFFFFF
            if bucket < audit_rate:
                reasons.append("deterministic_quality_audit")
        return reasons

    def deduplicate(self, detections):
        threshold = self.cfg["inference"]["canonical_dedup_iou"]
        candidates = []
        for det in detections:
            label = det["label"]
            canonical = self.canonical(label)
            candidates.append({
                **det,
                "canonical_label": canonical,
                "aliases": {label, canonical},
                "_weight": max(0.01, det["conf"]),
                "_primary_conf": det["conf"],
            })
        candidates.sort(key=lambda d: d["conf"], reverse=True)
        kept = []
        for det in candidates:
            match = next((x for x in kept if x["canonical_label"] == det["canonical_label"] and box_iou(x["box"], det["box"]) >= threshold), None)
            if match is None:
                kept.append(det)
                continue
            old_w, new_w = match["_weight"], det["_weight"]
            match["box"] = weighted_box(match["box"], old_w, det["box"], new_w)
            match["_weight"] = old_w + new_w
            match["conf"] = max(match["conf"], det["conf"])
            match["aliases"].update(det["aliases"])
            match["sources"] = sorted(set(match["sources"]) | set(det["sources"]))
            for source, score in det.get("source_scores", {}).items():
                match.setdefault("source_scores", {})[source] = max(
                    float(score), float(match.get("source_scores", {}).get(source, 0.0))
                )
            if match.get("mask_crop") is None and det.get("mask_crop") is not None:
                match["mask_crop"] = det["mask_crop"]
                match["mask_box"] = det.get("mask_box")
            if det["conf"] > match.get("_primary_conf", -1):
                match["label"] = det["label"]
                match["_primary_conf"] = det["conf"]
        for det in kept:
            det["aliases"] = sorted(det["aliases"])
            det.pop("_weight", None)
            det.pop("_primary_conf", None)
        return kept

    def _document_base(self, entry, status="ok", error=None, fallback_info=None):
        doc_id = f"{entry['video_id']}:{entry['frame_name']}"
        url = self.public_url_fn(entry["r2_key"])
        doc = {
            "schema_version": self.cfg["metadata"]["schema_version"],
            "pipeline_version": self.cfg["pipeline_version"],
            "config_hash": self.cfg["config_hash"],
            "manifest_hash": self.cfg["manifest_hash"],
            "runtime_hash": self.cfg["runtime_hash"],
            "resolved_detector": self.cfg["runtime"]["resolved_detector"],
            "weight_sha256": self.cfg["runtime"]["weight_sha256"],
            "text_encoder_sha256": self.cfg["runtime"]["text_encoder_sha256"],
            "prompt_embeddings_sha256": self.cfg["runtime"]["prompt_embeddings_sha256"],
            "document_id": doc_id,
            "keyframe_id": doc_id,
            "video_id": entry["video_id"],
            "frame_name": entry["frame_name"],
            "batch": entry["batch"],
            "genre": entry["genre"],
            "r2_key": entry["r2_key"],
            "image_url": url,
            "input_etag": entry.get("etag", ""),
            "status": status,
            "fallback_status": (fallback_info or {}).get("status", "not_applicable"),
            "fallback_reasons": (fallback_info or {}).get("reasons", []),
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "pipeline": {
                "detector": self.yoloe_name,
                "detector_weight_sha256": self.weight_sha256,
                "text_encoder": self.cfg["runtime"]["text_encoder"],
                "text_encoder_sha256": self.cfg["runtime"]["text_encoder_sha256"],
                "prompt_embeddings_sha256": self.cfg["runtime"]["prompt_embeddings_sha256"],
                "segmentation_dtype_guard": self.cfg["runtime"]["segmentation_dtype_guard"],
                "clip_revision": self.cfg["runtime"]["clip_revision"],
                "clip_code_sha256": self.cfg["runtime"]["clip_code_sha256"],
                "runtime_hash": self.cfg["runtime_hash"],
                "gdino_mode": self.cfg["inference"]["gdino_mode"],
                "gdino_model": self.cfg["models"]["gdino_model"],
                "gdino_revision": self.cfg["models"]["gdino_revision"],
                "gdino_weight_sha256": self.cfg["runtime"]["fallback_weight_sha256"],
                "precision": self.cfg["runtime"]["precision"],
                "image_size": self.cfg["inference"]["image_size"],
                "detection_conf": self.cfg["inference"]["detection_conf"],
                "count_conf": self.cfg["inference"]["count_conf"],
                "color_space": self.cfg["metadata"]["color_space"],
                "color_palette_version": self.cfg["metadata"]["color_palette_version"],
            },
        }
        if entry.get("frame_index") is not None:
            doc["frame_index"] = int(entry["frame_index"])
        if error:
            doc["error"] = str(error)[:500]
        if (fallback_info or {}).get("error"):
            doc["fallback_error"] = str(fallback_info["error"])[:500]
        return doc

    def error_document(self, entry, error):
        doc = self._document_base(entry, status="error", error=error)
        doc.update({
            "width": 0, "height": 0, "detection_count": 0, "counted_detection_count": 0,
            "labels": [], "canonical_labels": [], "spatial_tokens": [], "count_tokens": [],
            "object_text": "", "object_counts": [], "detections": [], "crowd_present": False,
            "counts_are_estimates": True, "count_is_lower_bound": False,
            "count_uncertain_labels": self.cfg["metadata"]["countable_classes"],
            "count_notes": ["image_processing_error"],
        })
        return doc

    def _count_decision(self, det):
        canonical = det["canonical_label"]
        if canonical not in set(self.cfg["metadata"]["countable_classes"]):
            return False, False, None, "not_countable"
        thresholds = self.cfg["metadata"]["count_thresholds"]
        scores = det.get("source_scores") or {s: det["conf"] for s in det.get("sources", [])}
        margins = []
        for source, score in scores.items():
            source_thresholds = thresholds.get(source, {})
            threshold = float(source_thresholds.get(canonical, source_thresholds.get("_default", self.cfg["inference"]["count_conf"])))
            margins.append((float(score) - threshold, source, threshold))
        if not margins:
            threshold = float(self.cfg["inference"]["count_conf"])
            return True, bool(det["conf"] >= threshold), threshold, "global_fallback"
        margin, source, threshold = max(margins)
        return True, bool(margin >= 0), threshold, source

    def build_document(self, entry, image, raw_detections, fallback_info=None):
        width, height = image.size
        min_area = self.cfg["inference"]["min_box_area_norm"]
        grid_size = self.cfg["metadata"]["grid_size"]
        detections = []
        for det in self.deduplicate(raw_detections):
            spatial = position_metadata(det["box"], width, height, grid_size)
            if spatial["size_norm"]["area"] < min_area:
                continue
            canonical = det["canonical_label"]
            color_meta = self.color.extract(
                image, det.get("mask_box") or det["box"], canonical, det.get("mask_crop"), det["conf"]
            )
            aliases = sorted(set(det["aliases"]) | set(self.cfg["vi_aliases"].get(canonical, [])))
            countable, counted, threshold_used, count_source = self._count_decision(det)
            item = {
                "instance_id": "",  # gán sau khi sort
                "label": det["label"],
                "canonical_label": canonical,
                "aliases": aliases,
                "label_group": self.cfg["label_groups"].get(canonical, "other"),
                "conf": round(float(det["conf"]), 5),
                "countable": countable,
                "counted": counted,
                "count_threshold_used": threshold_used,
                "count_source": count_source,
                "sources": sorted(det["sources"]),
                "source_scores": {k: round(float(v), 5) for k, v in sorted(det.get("source_scores", {}).items())},
                **color_meta,
                **spatial,
            }
            detections.append(item)
        detections.sort(key=lambda d: (-d["conf"], d["canonical_label"], d["bbox_norm"]["x1"]))
        for i, det in enumerate(detections):
            det["instance_id"] = f"{entry['video_id']}:{entry['frame_name']}:{i:03d}"

        counted = [d for d in detections if d["counted"]]
        grouped = defaultdict(list)
        for det in counted:
            grouped[det["canonical_label"]].append(det)
        crowd_present = any(d["canonical_label"] == "crowd" for d in detections)
        # max_det áp dụng cho YOLOE; đếm raw YOLO trước dedup/filter để không bỏ sót saturation.
        saturated = sum("yoloe" in d.get("sources", []) for d in raw_detections) >= self.cfg["inference"]["max_detections"]
        countable_classes = set(self.cfg["metadata"]["countable_classes"])
        uncertain_by_label = defaultdict(list)
        if crowd_present:
            uncertain_by_label["person"].append("crowd_present")
        if saturated:
            for label in countable_classes:
                uncertain_by_label[label].append("max_detections_reached")
        count_uncertain_labels = sorted(uncertain_by_label)
        count_is_lower_bound = bool(count_uncertain_labels)
        object_counts = []
        for label, items in sorted(grouped.items()):
            observed_aliases = sorted({alias for item in items for alias in item["aliases"]})
            object_counts.append({
                "label": label,
                "count": len(items),
                "countable": True,
                "max_conf": round(max(x["conf"] for x in items), 5),
                "mean_conf": round(sum(x["conf"] for x in items) / len(items), 5),
                "aliases": observed_aliases,
                "is_estimate": True,
                "is_lower_bound": bool(uncertain_by_label.get(label)),
                "lower_bound_reasons": uncertain_by_label.get(label, []),
                "count_quality": "lower_bound" if uncertain_by_label.get(label) else "detector_exact_candidate",
                "thresholds_used": sorted({round(float(x["count_threshold_used"]), 5) for x in items}),
                "sources": sorted({x["count_source"] for x in items}),
            })

        labels = sorted({d["label"] for d in detections})
        canonical_labels = sorted({d["canonical_label"] for d in detections})
        spatial_tokens = sorted({f"{d['canonical_label']}@{p}" for d in detections for p in d["position_tags"]})
        count_tokens = sorted(f"{x['label']}:{x['count']}" for x in object_counts)
        count_phrases = [f"{x['count']} {x['label']}" for x in object_counts]
        spatial_phrases = []
        seen_spatial = set()
        for d in counted:
            # Không để bbox fallback/ambiguous color làm nhiễu BM25 object_text.
            color_prefix = f"{d['dominant_color']} " if d.get("color_reliable") else ""
            phrase = f"{color_prefix}{d['canonical_label']} at {d['position']}"
            if phrase not in seen_spatial:
                seen_spatial.add(phrase)
                spatial_phrases.append(phrase)
        vi_terms = sorted({term for d in detections for term in self.cfg["vi_aliases"].get(d["canonical_label"], [])})
        object_text = "; ".join(count_phrases + spatial_phrases + vi_terms)

        notes = ["visible detector instances; occluded objects may be missed"]
        if crowd_present:
            notes.append("crowd detected: person count is a lower bound, not crowd population")
        if saturated:
            notes.append("max_detections reached: counts may be truncated")

        doc = self._document_base(entry, fallback_info=fallback_info)
        doc.update({
            "width": width, "height": height,
            "detection_count": len(detections), "counted_detection_count": len(counted),
            "labels": labels, "canonical_labels": canonical_labels,
            "spatial_tokens": spatial_tokens, "count_tokens": count_tokens,
            "object_text": object_text, "object_counts": object_counts, "detections": detections,
            "crowd_present": crowd_present, "counts_are_estimates": True,
            "count_is_lower_bound": count_is_lower_bound,
            "count_uncertain_labels": count_uncertain_labels, "count_notes": notes,
        })
        return doc

    def process_entries(self, entries):
        output = []
        by_genre = defaultdict(list)
        for entry in entries:
            by_genre[entry["genre"]].append(entry)

        for genre, genre_entries in sorted(by_genre.items()):
            for batch_entries in batched(genre_entries, self.cfg["inference"]["batch_size"]):
                valid_entries, images = [], []
                for entry in batch_entries:
                    try:
                        with Image.open(entry["local_path"]) as raw:
                            image = raw.convert("RGB")
                        valid_entries.append(entry)
                        images.append(image)
                    except Exception as exc:
                        output.append(self.error_document(entry, repr(exc)))
                if not images:
                    continue
                try:
                    yolo_dets = self.predict_yolo_batch(images, genre)
                    merged = [list(x) for x in yolo_dets]
                    fallback_meta = []
                    for dets, entry in zip(yolo_dets, valid_entries):
                        if self.gdino_failed:
                            fallback_meta.append({"status": "disabled_after_errors", "reasons": ["gdino_runtime_circuit_open"]})
                        else:
                            reasons = self.gdino_reasons(
                                dets, genre, f"{entry['video_id']}:{entry['frame_name']}"
                            )
                            fallback_meta.append({"status": "planned" if reasons else "skipped", "reasons": reasons})
                    fallback_ids = [i for i, meta in enumerate(fallback_meta) if meta["status"] == "planned"]
                    for id_batch in batched(fallback_ids, self.cfg["inference"]["gdino_batch_size"]):
                        gd_images = [images[i] for i in id_batch]
                        try:
                            gd_results = self.predict_gdino_batch(gd_images, genre)
                            self.gdino_failure_streak = 0
                            for image_i, gd_dets in zip(id_batch, gd_results):
                                merged[image_i].extend(gd_dets)
                                fallback_meta[image_i]["status"] = "used"
                        except Exception as gdino_exc:
                            # Retry batch nhỏ tới từng frame; không làm mất YOLOE result đã thành công.
                            failed = []
                            if len(id_batch) > 1:
                                for image_i in id_batch:
                                    try:
                                        one = self.predict_gdino_batch([images[image_i]], genre)[0]
                                        merged[image_i].extend(one)
                                        fallback_meta[image_i]["status"] = "used_after_single_retry"
                                    except Exception as one_exc:
                                        failed.append((image_i, one_exc))
                            else:
                                failed = [(id_batch[0], gdino_exc)]
                            for image_i, one_exc in failed:
                                fallback_meta[image_i]["status"] = "error"
                                fallback_meta[image_i]["error"] = repr(one_exc)[:300]
                            if failed and len(failed) == len(id_batch):
                                self.gdino_failure_streak += 1
                            else:
                                self.gdino_failure_streak = 0
                            if self.gdino_failure_streak >= 3:
                                self.gdino_failed = True
                            print(
                                f"Grounding DINO failed {len(failed)}/{len(id_batch)}; keep YOLOE-only:",
                                repr(gdino_exc)[:200],
                            )
                    for entry, image, dets, fallback_info in zip(valid_entries, images, merged, fallback_meta):
                        output.append(self.build_document(entry, image, dets, fallback_info=fallback_info))
                except Exception as exc:
                    # Sau khi ảnh đã decode, lỗi ở YOLO/metadata thường là lỗi hệ thống áp dụng cho
                    # cả batch (dependency, model state, CUDA, schema). Fail-fast để không đốt A100
                    # và không upload hàng nghìn error document như một retry shard giả hữu ích.
                    first_id = f"{valid_entries[0]['video_id']}:{valid_entries[0]['frame_name']}"
                    raise RuntimeError(
                        f"Inference batch failed genre={genre} first={first_id} size={len(valid_entries)}: {exc!r}"
                    ) from exc
                finally:
                    for image in images:
                        image.close()
        output.sort(key=lambda d: (d["video_id"], d.get("frame_index", 10**12), d["frame_name"]))
        return output


In [ ]:
%%writefile od_runner.py
import gzip
import hashlib
import importlib.metadata
import io
import json
import math
import platform
import random
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import torch
from PIL import Image, ImageDraw

from od_pipeline_lib import DetectionPipeline
from od_r2 import R2Store, iter_gzip_jsonl_stream, write_gzip_jsonl


def load_cfg(path="/content/aic26_od/od_config.json"):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def prepare_manifest(cfg, store=None):
    store = store or R2Store(cfg)
    store.check()
    rows = store.list_keyframes()
    if not rows:
        raise RuntimeError(f"Không thấy keyframe dưới {cfg['r2']['input_prefixes']}")
    manifest_fingerprint = hashlib.sha256()
    for row in rows:
        manifest_fingerprint.update(
            f"{row['r2_key']}\t{row.get('size',0)}\t{row.get('etag','')}\n".encode("utf-8")
        )
    manifest_hash = manifest_fingerprint.hexdigest()[:12]
    cfg["manifest_hash"] = manifest_hash
    cfg["manifest_count"] = len(rows)
    # Final prefix chỉ được bind sau khi load/checksum model và fingerprint runtime.
    cfg["r2"]["output_prefix"] = None
    out_dir = Path(cfg["local"]["output"]) / cfg["config_hash"]
    out_dir.mkdir(parents=True, exist_ok=True)
    local = write_gzip_jsonl(out_dir / "manifest.jsonl.gz", rows)
    cfg["local_manifest_path"] = str(local)
    config_path = Path("/content/aic26_od/od_config.json")
    config_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Manifest:", len(rows), "keyframes | hash:", manifest_hash, "| waiting for runtime binding")
    return rows


def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def bind_runtime(cfg, detector, manifest, store):
    code_files = ["od_pipeline_lib.py", "od_runner.py", "od_r2.py"]
    code_hashes = {}
    for name in code_files:
        path = Path(name)
        if not path.is_file():
            raise RuntimeError(f"Thiếu runtime source {name}; không thể tạo runtime_hash")
        code_hashes[name] = _sha256_file(path)
    distributions = [
        "ultralytics", "transformers", "torch", "torchvision", "numpy", "Pillow",
        "boto3", "botocore", "requests", "ftfy", "clip",
    ]
    versions = {}
    for name in distributions:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "missing"
    environment_lock = sorted(
        f"{(dist.metadata.get('Name') or 'unknown').lower()}=={dist.version}"
        for dist in importlib.metadata.distributions()
    )
    runtime = {
        "resolved_detector": detector.yoloe_name,
        "weight_sha256": detector.weight_sha256,
        "text_encoder": Path(detector.text_encoder_path).name,
        "text_encoder_sha256": detector.text_encoder_sha256,
        "prompt_embeddings_sha256": detector.prompt_embeddings_sha256,
        "segmentation_dtype_guard": detector.segmentation_dtype_guard,
        "clip_revision": cfg["models"]["clip_revision"],
        "clip_code_sha256": detector.clip_code_sha256,
        "resolved_fallback": cfg["models"]["gdino_model"],
        "fallback_revision": cfg["models"]["gdino_revision"],
        "fallback_weight_sha256": getattr(
            detector, "gdino_weight_sha256", cfg["models"]["gdino_weight_sha256"]
        ),
        "precision": detector.yolo_precision,
        "fallback_precision": (
            str(next(detector.gdino.parameters()).dtype).replace("torch.", "")
            if detector.gdino is not None else "not_loaded"
        ),
        "python": platform.python_version(),
        "packages": versions,
        "environment_lock_sha256": hashlib.sha256("\n".join(environment_lock).encode("utf-8")).hexdigest(),
        "torch_cuda": str(torch.version.cuda),
        "cudnn": str(torch.backends.cudnn.version()),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "code_sha256": code_hashes,
    }
    runtime_hash = hashlib.sha256(
        json.dumps(runtime, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    ).hexdigest()[:16]
    previous = cfg.get("runtime_hash")
    if previous and previous != runtime_hash:
        raise RuntimeError(f"Runtime đổi trong cùng session: {previous} -> {runtime_hash}")
    cfg["runtime"] = runtime
    cfg["runtime_hash"] = runtime_hash
    cfg["r2"]["output_prefix"] = (
        f"{cfg['r2']['output_prefix_base']}/input-{cfg['manifest_hash']}/runtime-{runtime_hash}"
    )
    config_path = Path("/content/aic26_od/od_config.json")
    config_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
    manifest_path = Path(cfg["local_manifest_path"])
    store.upload_file(
        manifest_path, f"{cfg['r2']['output_prefix']}/manifest.jsonl.gz",
        content_type="application/x-ndjson", content_encoding="gzip",
    )
    store.upload_file(config_path, f"{cfg['r2']['output_prefix']}/config.json", content_type="application/json")
    print("Runtime:", runtime_hash, "| detector:", detector.yoloe_name, "|", runtime["precision"])
    print("Final R2 output:", cfg["r2"]["output_prefix"])
    return runtime


def shard_key(cfg, shard_id):
    return f"{cfg['r2']['output_prefix']}/shards/part-{shard_id:06d}.jsonl.gz"


def completion_key(cfg, shard_id):
    return f"{cfg['r2']['output_prefix']}/complete/part-{shard_id:06d}.complete.json"


def existing_shards(cfg, store, manifest=None):
    """Lightweight resume audit; marker hỏng/mismatch bị coi là pending và tự overwrite."""
    prefix = f"{cfg['r2']['output_prefix']}/complete/"
    completed = set()
    for obj in store.list_keys(prefix):
        name = Path(obj["Key"]).name
        if not (name.startswith("part-") and name.endswith(".complete.json")):
            continue
        sid = int(name[5:11])
        problems = []
        try:
            marker = _read_json_object(store, obj["Key"])
        except Exception as exc:
            marker = {}
            problems.append(f"unreadable:{exc!r}")
        for field in ("config_hash", "manifest_hash", "runtime_hash"):
            if marker.get(field) != cfg.get(field):
                problems.append(f"{field}_mismatch")
        if marker.get("resolved_detector") != cfg.get("runtime", {}).get("resolved_detector"):
            problems.append("resolved_detector_mismatch")
        if marker.get("weight_sha256") != cfg.get("runtime", {}).get("weight_sha256"):
            problems.append("weight_sha256_mismatch")
        if marker.get("text_encoder_sha256") != cfg.get("runtime", {}).get("text_encoder_sha256"):
            problems.append("text_encoder_sha256_mismatch")
        if marker.get("prompt_embeddings_sha256") != cfg.get("runtime", {}).get("prompt_embeddings_sha256"):
            problems.append("prompt_embeddings_sha256_mismatch")
        if marker.get("segmentation_dtype_guard") != cfg.get("runtime", {}).get("segmentation_dtype_guard"):
            problems.append("segmentation_dtype_guard_mismatch")
        if marker.get("fallback_weight_sha256") != cfg.get("runtime", {}).get("fallback_weight_sha256"):
            problems.append("fallback_weight_sha256_mismatch")
        if marker.get("error_count") != 0:
            problems.append("error_count_nonzero")
        if not store.object_exists(shard_key(cfg, sid)):
            problems.append("raw_shard_missing")
        if manifest is not None:
            shard_size = cfg["run"]["shard_size"]
            rows = manifest[sid * shard_size:(sid + 1) * shard_size]
            slice_hash = hashlib.sha256("\n".join(x["r2_key"] for x in rows).encode("utf-8")).hexdigest()
            if marker.get("manifest_slice_hash") != slice_hash:
                problems.append("manifest_slice_hash_mismatch")
            if marker.get("document_count") != len(rows):
                problems.append("document_count_mismatch")
        if problems:
            print(f"Resume audit: shard {sid:06d} invalid -> retry ({', '.join(problems)})")
        else:
            completed.add(sid)
    return completed


def run_pipeline(cfg, manifest=None, max_new_shards=None):
    store = R2Store(cfg)
    manifest = manifest or prepare_manifest(cfg, store)
    # Load/checksum required model trước khi quyết định namespace/resume.
    detector = DetectionPipeline(cfg, public_url_fn=store.public_url)
    bind_runtime(cfg, detector, manifest, store)
    shard_size = cfg["run"]["shard_size"]
    shards = [(i // shard_size, manifest[i:i + shard_size]) for i in range(0, len(manifest), shard_size)]
    existing = existing_shards(cfg, store, manifest=manifest)
    pending = [(sid, rows) for sid, rows in shards if sid not in existing]
    if max_new_shards is not None:
        pending = pending[:max_new_shards]
    print("Shards total:", len(shards), "| completed:", len(existing), "| pending this run:", len(pending))
    if not pending:
        return {"status": "already_complete", "shards": len(shards), "frames": len(manifest)}

    # Bắt đầu tải shard đầu sau khi runtime namespace đã khóa.
    prefetch_pool = ThreadPoolExecutor(max_workers=1)
    future = prefetch_pool.submit(store.stage_entries, pending[0][1])
    out_dir = Path(cfg["local"]["output"]) / cfg["config_hash"] / cfg["runtime_hash"] / "shards"
    out_dir.mkdir(parents=True, exist_ok=True)
    started = time.time()
    processed_frames = 0
    status_counts = Counter()
    committed_new_shards, retry_shards, retry_frame_ids = 0, [], []

    try:
        for pos, (sid, rows) in enumerate(pending):
            staged = future.result()
            if pos + 1 < len(pending):
                future = prefetch_pool.submit(store.stage_entries, pending[pos + 1][1])
            t0 = time.time()
            docs = detector.process_entries(staged)
            if len(docs) != len(rows):
                raise RuntimeError(f"Shard {sid}: output {len(docs)} != input {len(rows)}")
            local = write_gzip_jsonl(out_dir / f"part-{sid:06d}.jsonl.gz", docs)
            remote = shard_key(cfg, sid)
            store.upload_file(local, remote, content_type="application/x-ndjson", content_encoding="gzip")
            shard_status = Counter(d["status"] for d in docs)
            status_counts.update(shard_status)
            sha256 = hashlib.sha256(local.read_bytes()).hexdigest()
            marker = {
                "shard_id": sid, "document_count": len(docs), "error_count": shard_status.get("error", 0),
                "sha256": sha256, "manifest_hash": cfg["manifest_hash"],
                "runtime_hash": cfg["runtime_hash"],
                "resolved_detector": cfg["runtime"]["resolved_detector"],
                "weight_sha256": cfg["runtime"]["weight_sha256"],
                "text_encoder_sha256": cfg["runtime"]["text_encoder_sha256"],
                "prompt_embeddings_sha256": cfg["runtime"]["prompt_embeddings_sha256"],
                "segmentation_dtype_guard": cfg["runtime"]["segmentation_dtype_guard"],
                "fallback_weight_sha256": cfg["runtime"]["fallback_weight_sha256"],
                "manifest_slice_hash": hashlib.sha256(
                    "\n".join(x["r2_key"] for x in rows).encode("utf-8")
                ).hexdigest(),
                "config_hash": cfg["config_hash"], "first_document_id": docs[0]["document_id"],
                "last_document_id": docs[-1]["document_id"], "committed_at": time.time(),
            }
            marker_path = out_dir / f"part-{sid:06d}.complete.json"
            marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8")
            if marker["error_count"] == 0:
                # Marker upload là commit cuối cùng. Resume bỏ qua duy nhất shard có marker này.
                store.upload_file(marker_path, completion_key(cfg, sid), content_type="application/json")
                committed_new_shards += 1
            else:
                failed_ids = [d["document_id"] for d in docs if d.get("status") == "error"]
                retry_shards.append(sid)
                retry_frame_ids.extend(failed_ids)
                retry = {
                    **marker,
                    "retry_frame_ids": failed_ids,
                    "reason": "not_committed_due_to_error_documents",
                }
                retry_path = out_dir / f"part-{sid:06d}.retry.json"
                retry_path.write_text(json.dumps(retry, ensure_ascii=False, indent=2), encoding="utf-8")
                store.upload_file(
                    retry_path,
                    f"{cfg['r2']['output_prefix']}/retry/part-{sid:06d}.retry.json",
                    content_type="application/json",
                )
            processed_frames += len(docs)
            store.cleanup_entries(staged)
            if not cfg["run"].get("keep_local_shards"):
                local.unlink()
            elapsed = max(1e-9, time.time() - started)
            print(
                f"[{pos+1}/{len(pending)}] shard={sid:06d} frames={len(docs)} "
                f"status={dict(shard_status)} committed={marker['error_count'] == 0} "
                f"shard_sec={time.time()-t0:.1f} avg={processed_frames/elapsed:.2f} frame/s",
                flush=True,
            )
    finally:
        prefetch_pool.shutdown(wait=True, cancel_futures=True)

    summary = {
        "status": "finished", "attempted_shards": len(pending),
        "committed_new_shards": committed_new_shards, "new_frames": processed_frames,
        "retry_shards": retry_shards, "retry_frame_ids": retry_frame_ids,
        "status_counts": dict(status_counts), "elapsed_seconds": round(time.time() - started, 2),
        "config_hash": cfg["config_hash"], "manifest_hash": cfg["manifest_hash"],
        "runtime_hash": cfg["runtime_hash"], "output_prefix": cfg["r2"]["output_prefix"],
    }
    summary_path = Path(cfg["local"]["output"]) / cfg["config_hash"] / cfg["runtime_hash"] / "last_run_summary.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    store.upload_file(summary_path, f"{cfg['r2']['output_prefix']}/last_run_summary.json", content_type="application/json")
    return summary


def _read_json_object(store, key):
    return json.loads(store.client.get_object(Bucket=store.bucket, Key=key)["Body"].read())


def validate_output(cfg, manifest=None, max_shards=None, sample_size=200):
    store = R2Store(cfg)
    manifest = manifest or prepare_manifest(cfg, store)
    shard_size = cfg["run"]["shard_size"]
    expected_shards = math.ceil(len(manifest) / shard_size)
    completed = existing_shards(cfg, store, manifest=manifest)
    raw_shard_ids = {
        int(Path(obj["Key"]).name[5:11])
        for obj in store.list_keys(f"{cfg['r2']['output_prefix']}/shards/")
        if Path(obj["Key"]).name.startswith("part-") and obj["Key"].endswith(".jsonl.gz")
    }
    selected_ids = sorted(completed)
    if max_shards is not None:
        selected_ids = selected_ids[:max_shards]
    expected_ids = {
        f"{x['video_id']}:{x['frame_name']}"
        for sid in selected_ids for x in manifest[sid * shard_size:(sid + 1) * shard_size]
    }
    stats = Counter()
    labels = Counter()
    genres = Counter()
    genre_labels = Counter()
    confidence_sum = Counter()
    confidence_min = defaultdict(lambda: 1.0)
    confidence_max = defaultdict(float)
    bad_examples = []
    errors, warnings = [], []
    manifest_doc_ids = [f"{x['video_id']}:{x['frame_name']}" for x in manifest]
    if len(manifest_doc_ids) != len(set(manifest_doc_ids)):
        errors.append(f"duplicate_manifest_document_ids:{len(manifest_doc_ids)-len(set(manifest_doc_ids))}")
    if raw_shard_ids - completed:
        warnings.append(f"uncommitted_retry_shards:{sorted(raw_shard_ids-completed)[:50]}")
    if completed - raw_shard_ids:
        errors.append(f"completion_marker_without_raw_shard:{sorted(completed-raw_shard_ids)[:50]}")
    seen_ids = set()
    sample_ids = []
    genre_sample_ids = defaultdict(list)
    label_sample_id = {}
    rng = random.Random(20260802)
    valid_sources = {"yoloe", "gdino"}
    palette = set(cfg["metadata"]["palette"])
    countable = set(cfg["metadata"]["countable_classes"])
    color_classes = set(cfg["metadata"]["color_classes"])
    marker_fingerprints = []
    for sid in selected_ids:
        key = shard_key(cfg, sid)
        marker = _read_json_object(store, completion_key(cfg, sid))
        expected_slice = manifest[sid * shard_size:(sid + 1) * shard_size]
        expected_slice_hash = hashlib.sha256(
            "\n".join(x["r2_key"] for x in expected_slice).encode("utf-8")
        ).hexdigest()
        if marker.get("manifest_hash") != cfg.get("manifest_hash") or marker.get("config_hash") != cfg.get("config_hash"):
            errors.append(f"marker_namespace_mismatch:{sid}")
        if marker.get("runtime_hash") != cfg.get("runtime_hash"):
            errors.append(f"marker_runtime_mismatch:{sid}")
        if marker.get("resolved_detector") != cfg.get("runtime", {}).get("resolved_detector"):
            errors.append(f"marker_detector_mismatch:{sid}")
        if marker.get("weight_sha256") != cfg.get("runtime", {}).get("weight_sha256"):
            errors.append(f"marker_weight_mismatch:{sid}")
        if marker.get("text_encoder_sha256") != cfg.get("runtime", {}).get("text_encoder_sha256"):
            errors.append(f"marker_text_encoder_mismatch:{sid}")
        if marker.get("prompt_embeddings_sha256") != cfg.get("runtime", {}).get("prompt_embeddings_sha256"):
            errors.append(f"marker_prompt_embeddings_mismatch:{sid}")
        if marker.get("segmentation_dtype_guard") != cfg.get("runtime", {}).get("segmentation_dtype_guard"):
            errors.append(f"marker_segmentation_dtype_guard_mismatch:{sid}")
        if marker.get("fallback_weight_sha256") != cfg.get("runtime", {}).get("fallback_weight_sha256"):
            errors.append(f"marker_fallback_weight_mismatch:{sid}")
        if marker.get("error_count") != 0:
            errors.append(f"committed_marker_has_errors:{sid}:{marker.get('error_count')}")
        if marker.get("manifest_slice_hash") != expected_slice_hash:
            errors.append(f"marker_manifest_slice_mismatch:{sid}")
        try:
            payload = store.client.get_object(Bucket=store.bucket, Key=key)["Body"].read()
        except Exception as exc:
            errors.append(f"marker_without_shard:{sid}:{exc!r}")
            continue
        if hashlib.sha256(payload).hexdigest() != marker.get("sha256"):
            errors.append(f"checksum_mismatch:{sid}")
        marker_fingerprints.append(f"{sid}:{marker.get('sha256','')}")
        shard_docs = 0
        for doc in iter_gzip_jsonl_stream(io.BytesIO(payload)):
            shard_docs += 1
            stats["documents"] += 1
            stats[f"status_{doc.get('status','missing')}"] += 1
            stats[f"fallback_{doc.get('fallback_status','missing')}"] += 1
            if doc.get("status") != "ok":
                errors.append(f"non_ok_document:{doc.get('document_id')}:{doc.get('status')}")
            if (
                doc.get("config_hash") != cfg.get("config_hash")
                or doc.get("manifest_hash") != cfg.get("manifest_hash")
                or doc.get("runtime_hash") != cfg.get("runtime_hash")
                or doc.get("resolved_detector") != cfg.get("runtime", {}).get("resolved_detector")
                or doc.get("weight_sha256") != cfg.get("runtime", {}).get("weight_sha256")
                or doc.get("text_encoder_sha256") != cfg.get("runtime", {}).get("text_encoder_sha256")
                or doc.get("prompt_embeddings_sha256") != cfg.get("runtime", {}).get("prompt_embeddings_sha256")
            ):
                errors.append(f"document_runtime_mismatch:{doc.get('document_id')}")
            stats["detections"] += len(doc.get("detections", []))
            if not doc.get("detections"):
                stats["zero_detection_documents"] += 1
            stats["counted_detections"] += doc.get("counted_detection_count", 0)
            genres[doc.get("genre", "missing")] += 1
            genre = doc.get("genre", "missing")
            doc_id = doc.get("document_id")
            if doc_id in seen_ids:
                errors.append(f"duplicate_document_id:{doc_id}")
            seen_ids.add(doc_id)
            if len(sample_ids) < sample_size:
                sample_ids.append(doc_id)
            else:
                j = rng.randrange(stats["documents"])
                if j < sample_size:
                    sample_ids[j] = doc_id
            genre_cap = max(5, sample_size // 20)
            genre_bucket = genre_sample_ids[genre]
            if len(genre_bucket) < genre_cap:
                genre_bucket.append(doc_id)
            else:
                j = rng.randrange(genres[genre])
                if j < genre_cap:
                    genre_bucket[j] = doc_id
            expected_counted = Counter()
            for det in doc.get("detections", []):
                label = det.get("canonical_label")
                labels[label] += 1
                genre_labels[f"{doc.get('genre','missing')}:{label}"] += 1
                label_sample_id.setdefault(label, doc_id)
                conf = float(det.get("conf", -1))
                confidence_sum[label] += conf
                confidence_min[label] = min(confidence_min[label], conf)
                confidence_max[label] = max(confidence_max[label], conf)
                b = det["bbox_norm"]
                if not (0 <= b["x1"] <= b["x2"] <= 1 and 0 <= b["y1"] <= b["y2"] <= 1):
                    stats["invalid_boxes"] += 1
                    if len(bad_examples) < 10:
                        bad_examples.append((doc["document_id"], det["instance_id"], b))
                if not (0 <= conf <= 1):
                    errors.append(f"invalid_confidence:{doc_id}:{det.get('instance_id')}")
                if not set(det.get("sources", [])) <= valid_sources:
                    errors.append(f"invalid_source:{doc_id}:{det.get('sources')}")
                color_method = det.get("color_method", "missing")
                stats[f"color_method_{color_method}"] += 1
                if label in color_classes:
                    if color_method.startswith("skipped_"):
                        if det.get("dominant_color") or det.get("color_names") or det.get("color_reliable"):
                            errors.append(f"skipped_color_has_value:{doc_id}:{label}")
                    elif det.get("dominant_color") not in palette:
                        errors.append(f"invalid_color:{doc_id}:{det.get('dominant_color')}")
                elif det.get("dominant_color") or det.get("color_names") or color_method != "skipped_not_color_class":
                    errors.append(f"non_color_class_has_color:{doc_id}:{label}")
                if not (0 <= float(det.get("color_confidence", -1)) <= 1):
                    errors.append(f"invalid_color_confidence:{doc_id}:{det.get('instance_id')}")
                if not (0 <= float(det.get("color_purity", -1)) <= 1):
                    errors.append(f"invalid_color_purity:{doc_id}:{det.get('instance_id')}")
                if not (0 <= float(det.get("color_entropy", -1)) <= 1):
                    errors.append(f"invalid_color_entropy:{doc_id}:{det.get('instance_id')}")
                if float(det.get("color_delta_e_p50", -1)) < 0:
                    errors.append(f"invalid_color_delta_e:{doc_id}:{det.get('instance_id')}")
                if not (0 <= float(det.get("color_coverage", -1)) <= 1):
                    errors.append(f"invalid_color_coverage:{doc_id}:{det.get('instance_id')}")
                if bool(det.get("countable")) != (label in countable):
                    errors.append(f"countable_mismatch:{doc_id}:{label}")
                if det.get("counted"):
                    expected_counted[label] += 1
                    if not det.get("countable"):
                        errors.append(f"non_countable_was_counted:{doc_id}:{label}")
            actual_counts = {x["label"]: x["count"] for x in doc.get("object_counts", [])}
            if dict(expected_counted) != actual_counts:
                errors.append(f"count_consistency:{doc_id}:{dict(expected_counted)}!={actual_counts}")
            if sum(actual_counts.values()) != doc.get("counted_detection_count"):
                errors.append(f"counted_total_mismatch:{doc_id}")
            uncertain = set(doc.get("count_uncertain_labels", []))
            if bool(uncertain) != bool(doc.get("count_is_lower_bound")):
                errors.append(f"root_count_uncertainty_mismatch:{doc_id}")
            for count_item in doc.get("object_counts", []):
                label = count_item.get("label")
                if label not in countable or count_item.get("countable") is not True:
                    errors.append(f"invalid_countable_count_item:{doc_id}:{label}")
                if bool(count_item.get("is_lower_bound")) != (label in uncertain):
                    errors.append(f"per_label_lower_bound_mismatch:{doc_id}:{label}")
                expected_quality = "lower_bound" if label in uncertain else "detector_exact_candidate"
                if count_item.get("count_quality") != expected_quality:
                    errors.append(f"count_quality_mismatch:{doc_id}:{label}")
        if shard_docs != marker.get("document_count"):
            errors.append(f"marker_document_count:{sid}:{shard_docs}!={marker.get('document_count')}")
    stats["shards"] = len(selected_ids)
    missing_ids = sorted(expected_ids - seen_ids)
    extra_ids = sorted(seen_ids - expected_ids)
    if missing_ids:
        errors.append(f"missing_document_ids:{len(missing_ids)} sample={missing_ids[:10]}")
    if extra_ids:
        errors.append(f"extra_document_ids:{len(extra_ids)} sample={extra_ids[:10]}")
    if max_shards is None:
        missing_shards = sorted(set(range(expected_shards)) - completed)
        if missing_shards:
            errors.append(f"missing_completion_markers:{len(missing_shards)} sample={missing_shards[:20]}")
    # Quality-collapse checks are warnings: thresholds depend on the collection.
    if stats["documents"]:
        zero_rate = stats["zero_detection_documents"] / stats["documents"]
        if zero_rate > 0.90 and stats["documents"] >= 100:
            warnings.append(f"zero_detection_collapse:rate={zero_rate:.3f}")
        dominant_share = labels.most_common(1)[0][1] / max(1, sum(labels.values())) if labels else 0
        if dominant_share > 0.85 and stats["documents"] >= 100:
            warnings.append(f"dominant_class_collapse:{labels.most_common(1)[0]} share={dominant_share:.3f}")
    confidence = {
        label: {
            "min": round(confidence_min[label], 5),
            "mean": round(confidence_sum[label] / count, 5),
            "max": round(confidence_max[label], 5),
        }
        for label, count in labels.items()
    }
    contact_ids = []
    for doc_id in list(label_sample_id.values()) + [x for bucket in genre_sample_ids.values() for x in bucket] + sample_ids:
        if doc_id not in contact_ids:
            contact_ids.append(doc_id)
        if len(contact_ids) >= sample_size:
            break
    result = {
        "ok": not errors, "validation_ok": not errors, "errors": errors[:200], "warnings": warnings,
        "stats": dict(stats), "genres": dict(genres), "top_labels": labels.most_common(30),
        "top_labels_by_genre": genre_labels.most_common(100),
        "confidence_by_label": dict(sorted(confidence.items())), "bad_examples": bad_examples,
        "sample_document_ids": contact_ids, "expected_documents": len(expected_ids),
        "completed_shards": len(completed), "committed_shards": len(completed),
        "expected_shards": expected_shards,
        "config_hash": cfg.get("config_hash"), "manifest_hash": cfg.get("manifest_hash"),
        "runtime_hash": cfg.get("runtime_hash"),
        "document_ids_sha256": hashlib.sha256("\n".join(sorted(seen_ids)).encode("utf-8")).hexdigest(),
        "marker_set_sha256": hashlib.sha256("\n".join(sorted(marker_fingerprints)).encode("utf-8")).hexdigest(),
        "validated_at": datetime.now(timezone.utc).isoformat(),
    }
    if max_shards is None:
        report_path = (
            Path(cfg["local"]["output"]) / cfg["config_hash"] / cfg["runtime_hash"]
            / "validation" / "release-report.json"
        )
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        store.upload_file(
            report_path, f"{cfg['r2']['output_prefix']}/validation/release-report.json",
            content_type="application/json",
        )
        result["release_report"] = str(report_path)
    return result


def build_contact_sheet(cfg, manifest, document_ids, columns=5, thumb=(320, 200)):
    """Tạo contact sheet deterministic có bbox/label/conf/count rồi upload cạnh validation."""
    store = R2Store(cfg)
    wanted = set(document_ids)
    docs = {}
    for sid in sorted(existing_shards(cfg, store)):
        payload = store.client.get_object(Bucket=store.bucket, Key=shard_key(cfg, sid))["Body"]
        for doc in iter_gzip_jsonl_stream(payload):
            if doc["document_id"] in wanted:
                docs[doc["document_id"]] = doc
        if len(docs) == len(wanted):
            break
    entries = {f"{x['video_id']}:{x['frame_name']}": x for x in manifest if f"{x['video_id']}:{x['frame_name']}" in docs}
    staged = store.stage_entries(list(entries.values()))
    rows = math.ceil(len(staged) / columns)
    canvas = Image.new("RGB", (columns * thumb[0], rows * thumb[1]), "white")
    draw = ImageDraw.Draw(canvas)
    for i, entry in enumerate(staged):
        doc_id = f"{entry['video_id']}:{entry['frame_name']}"
        doc = docs[doc_id]
        with Image.open(entry["local_path"]) as raw:
            image = raw.convert("RGB")
        image.thumbnail((thumb[0], thumb[1] - 24), Image.Resampling.LANCZOS)
        sx, sy = image.width / doc["width"], image.height / doc["height"]
        tile = Image.new("RGB", thumb, "white")
        tile.paste(image, (0, 20))
        td = ImageDraw.Draw(tile)
        td.text((3, 3), f"{doc_id} | counts: " + ", ".join(doc.get("count_tokens", []))[:120], fill="black")
        for det in doc.get("detections", []):
            b = det["bbox_px"]
            box = (b["x1"] * sx, 20 + b["y1"] * sy, b["x2"] * sx, 20 + b["y2"] * sy)
            td.rectangle(box, outline="red", width=2)
            color = det.get("dominant_color") or "no-color"
            td.text((box[0] + 2, box[1] + 2), f"{det['canonical_label']} {det['conf']:.2f} {color}", fill="yellow", stroke_width=2, stroke_fill="black")
        canvas.paste(tile, ((i % columns) * thumb[0], (i // columns) * thumb[1]))
        image.close()
    out = Path(cfg["local"]["output"]) / cfg["config_hash"] / cfg["runtime_hash"] / "validation" / "contact-sheet.jpg"
    out.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out, quality=90)
    store.upload_file(out, f"{cfg['r2']['output_prefix']}/validation/contact-sheet.jpg", content_type="image/jpeg")
    store.cleanup_entries(staged)
    return str(out)


def evaluate_ground_truth(cfg, ground_truth_jsonl, iou_threshold=0.5, per_frame_k=5):
    """Detection/count metrics. GT phải annotate đầy đủ các class được evaluate trong mỗi frame."""
    from od_pipeline_lib import box_iou
    gt_docs = {}
    with open(ground_truth_jsonl, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                gt_docs[row["document_id"]] = row.get("detections", [])
    store, pred_docs = R2Store(cfg), {}
    for sid in sorted(existing_shards(cfg, store)):
        body = store.client.get_object(Bucket=store.bucket, Key=shard_key(cfg, sid))["Body"]
        for doc in iter_gzip_jsonl_stream(body):
            if doc["document_id"] in gt_docs:
                pred_docs[doc["document_id"]] = doc.get("detections", [])
        if len(pred_docs) == len(gt_docs):
            break
    tp_by, fp_by, fn_by = Counter(), Counter(), Counter()
    count_error_by, count_samples_by = Counter(), Counter()
    box_hits = gt_total = frame_count_exact = presence_correct = presence_total = 0
    eval_classes = set(cfg["metadata"]["countable_classes"])
    for doc_id, gt in gt_docs.items():
        pred = pred_docs.get(doc_id, [])
        gt_counts, pred_counts = Counter(x["canonical_label"] for x in gt), Counter(x["canonical_label"] for x in pred if x.get("counted"))
        frame_count_exact += int(all(gt_counts[label] == pred_counts[label] for label in eval_classes))
        for label in eval_classes:
            count_error_by[label] += abs(gt_counts[label] - pred_counts[label])
            count_samples_by[label] += 1
            presence_correct += int(bool(gt_counts[label]) == bool(pred_counts[label]))
            presence_total += 1
        for label in {x["canonical_label"] for x in gt} | {x["canonical_label"] for x in pred}:
            gt_items = [(i, x) for i, x in enumerate(gt) if x["canonical_label"] == label]
            pred_items = [(j, x) for j, x in enumerate(pred) if x["canonical_label"] == label]
            pairs = []
            for gi, truth in gt_items:
                tb = truth["bbox_norm"]
                ta = [tb[k] for k in ("x1", "y1", "x2", "y2")]
                for pj, detection in pred_items:
                    pb = detection["bbox_norm"]
                    iou = box_iou(ta, [pb[k] for k in ("x1", "y1", "x2", "y2")])
                    if iou >= iou_threshold:
                        pairs.append((iou, gi, pj))
            # Global descending-IoU assignment avoids dependence on annotation order.
            used_gt, used_pred = set(), set()
            for _, gi, pj in sorted(pairs, reverse=True):
                if gi not in used_gt and pj not in used_pred:
                    used_gt.add(gi)
                    used_pred.add(pj)
            tp_by[label] += len(used_gt)
            fn_by[label] += len(gt_items) - len(used_gt)
            fp_by[label] += len(pred_items) - len(used_pred)
        for truth in gt:
            gt_total += 1
            tb = truth["bbox_norm"]
            ta = [tb["x1"], tb["y1"], tb["x2"], tb["y2"]]
            ranked = sorted(
                [d for d in pred if d["canonical_label"] == truth["canonical_label"]],
                key=lambda d: d["conf"], reverse=True,
            )[:per_frame_k]
            if any(box_iou(ta, [d["bbox_norm"][k] for k in ("x1", "y1", "x2", "y2")]) >= iou_threshold for d in ranked):
                box_hits += 1
    classes = sorted(set(tp_by) | set(fp_by) | set(fn_by))
    per_class = {}
    for label in classes:
        tp, fp, fn = tp_by[label], fp_by[label], fn_by[label]
        precision, recall = tp / max(1, tp + fp), tp / max(1, tp + fn)
        per_class[label] = {
            "precision": precision, "recall": recall,
            "f1": 2 * precision * recall / max(1e-12, precision + recall),
            "tp": tp, "fp": fp, "fn": fn,
        }
    tp, fp, fn = sum(tp_by.values()), sum(fp_by.values()), sum(fn_by.values())
    precision, recall = tp / max(1, tp + fp), tp / max(1, tp + fn)
    return {
        "documents": len(gt_docs), "precision": precision, "recall": recall,
        "f1": 2 * precision * recall / max(1e-12, precision + recall),
        "per_class_detection": per_class,
        "count_mae_by_class": {label: count_error_by[label] / max(1, count_samples_by[label]) for label in sorted(eval_classes)},
        "count_exact_match_accuracy_by_frame": frame_count_exact / max(1, len(gt_docs)),
        "presence_absence_accuracy": presence_correct / max(1, presence_total),
        f"per_frame_box_recall_at_{per_frame_k}": box_hits / max(1, gt_total),
        "metric_note": "per_frame_box_recall_at_k is detector QA, not cross-frame spatial retrieval",
        "tp": tp, "fp": fp, "fn": fn,
        "missing_prediction_documents": sorted(set(gt_docs) - set(pred_docs)),
    }


In [ ]:
# === 6. Kết nối R2 + tạo manifest ổn định ===
import json
from od_r2 import R2Store
from od_runner import load_cfg, prepare_manifest

cfg = load_cfg()
r2 = R2Store(cfg)
r2.check()
manifest = prepare_manifest(cfg, r2)

print("Videos:", len({x['video_id'] for x in manifest}))
print("Frames:", len(manifest))
print("First:", {k: manifest[0][k] for k in ('r2_key','video_id','frame_name','genre')})
print("Last :", {k: manifest[-1][k] for k in ('r2_key','video_id','frame_name','genre')})


In [ ]:
# === 7. OPTIONAL smoke-test một shard ===
# Đặt True khi muốn kiểm model/metadata trước. Shard smoke được upload đúng namespace;
# cell full phía sau sẽ resume và không chạy lại shard này.
RUN_SMOKE_TEST = True  # fail-fast trên shard đầu; nếu pass, cell full kế tiếp resume 373 shard còn lại

if RUN_SMOKE_TEST:
    from od_runner import run_pipeline
    smoke_summary = run_pipeline(cfg, manifest=manifest, max_new_shards=1)
    print(json.dumps(smoke_summary, ensure_ascii=False, indent=2))
else:
    print("Bỏ qua smoke-test. Đặt RUN_SMOKE_TEST=True nếu cần.")


In [ ]:
# === 8. Chạy full A100; tự resume theo immutable shard trên R2 ===
from od_runner import run_pipeline

if cfg["run"]["full"]:
    summary = run_pipeline(cfg, manifest=manifest)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
else:
    print("RUN_FULL=False: không chạy full.")


In [ ]:
# === 9. OPTIONAL: bổ sung Grounding DINO FP32 sau khi YOLOE đã chạy xong ===
# Cell này KHÔNG chạy lại YOLOE. Nó đọc shard YOLOE đã commit, tính lại fallback policy từ
# detection đã lưu, chỉ tải/infer các frame cần GDINO, rồi ghi một namespace v5.1 đầy đủ mới.
# Marker theo shard giúp cell tự resume nếu Colab/R2 bị ngắt giữa chừng.
import copy
import gc
import gzip
import hashlib
import io
import json
import math
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import torch
from PIL import Image

from od_pipeline_lib import ColorExtractor, batched, box_iou, position_metadata, weighted_box
from od_r2 import R2Store, iter_gzip_jsonl_stream, write_gzip_jsonl
from od_runner import completion_key, existing_shards, shard_key


RUN_GDINO_POSTHOC = True
# Cùng Colab runtime sau cell 8: để None. Nếu runtime đã restart, dán exact "Final R2 output"
# của run YOLOE vào đây; cell sẽ đọc source config từ R2 và không load/chạy lại YOLOE.
GDINO_SOURCE_PREFIX = None
# Ví dụ:
# GDINO_SOURCE_PREFIX = "Derived/ObjectDetection/aic26-od-v5/7af657b02cb0/input-6bbfa222abe6/runtime-cf50c70a8c3b6e43"
GDINO_POSTHOC_BATCH_SIZE = 4       # FP32; tăng thận trọng. Code tự chia đôi nếu CUDA OOM.
GDINO_POSTHOC_DOWNLOAD_WORKERS = 32
GDINO_POSTHOC_CODE_VERSION = "aic26-gdino-posthoc-fp32-v1"


def _stable_hash(value, length=None):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    digest = hashlib.sha256(payload).hexdigest()
    return digest[:length] if length else digest


def _r2_payload(store, key, attempts=6):
    # Retry toàn bộ object khi stream R2 timeout giữa chừng; không resume từ payload cụt.
    for attempt in range(attempts):
        try:
            body = store.client.get_object(Bucket=store.bucket, Key=key)["Body"]
            try:
                return body.read()
            finally:
                body.close()
        except Exception:
            if attempt + 1 == attempts:
                raise
            time.sleep(min(20.0, 1.5 * (2 ** attempt)))


def _read_shard(store, key):
    return list(iter_gzip_jsonl_stream(io.BytesIO(_r2_payload(store, key))))


def _labels_for_genre(run_cfg, genre):
    labels, seen = [], set()
    for label in run_cfg["vocab"]["base"] + run_cfg["vocab"]["genres"].get(genre, []):
        if label not in seen:
            labels.append(label)
            seen.add(label)
    return labels


def _canonical(run_cfg, label):
    return run_cfg["canonical_map"].get(label, label)


def _posthoc_reasons(run_cfg, doc):
    # Không dùng fallback_status=disabled_after_errors: circuit breaker đã gán trạng thái đó
    # cho gần như mọi frame. Policy phải được tính lại từ YOLO detection đã lưu.
    detections = doc.get("detections", [])
    if any("gdino" in det.get("sources", []) for det in detections):
        return []
    if not detections:
        return ["no_yolo_detection"]
    inf = run_cfg["inference"]
    reasons = []
    if max(float(det.get("conf", 0.0)) for det in detections) < float(inf["gdino_trigger_top_conf"]):
        reasons.append("low_top_confidence")
    minimum = int(inf.get("gdino_min_detections_by_genre", {}).get(doc["genre"], 1))
    if len(detections) < minimum:
        reasons.append(f"low_detection_count_lt_{minimum}")
    anchors = set(inf.get("gdino_anchor_classes_by_genre", {}).get(doc["genre"], []))
    found = {
        label
        for det in detections
        for label in (det.get("label", ""), det.get("canonical_label", ""))
        if label
    }
    if anchors and not (anchors & found):
        reasons.append("missing_genre_anchor_class")
    audit_rate = float(inf.get("gdino_audit_rate_by_genre", {}).get(doc["genre"], 0.0))
    if audit_rate > 0:
        bucket = int(hashlib.sha256(doc["document_id"].encode("utf-8")).hexdigest()[:8], 16) / 0xFFFFFFFF
        if bucket < audit_rate:
            reasons.append("deterministic_quality_audit")
    return reasons


class _GroundingDINOFP32:
    def __init__(self, run_cfg):
        from huggingface_hub import hf_hub_download
        from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

        self.cfg = run_cfg
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        if self.device == "cpu":
            raise RuntimeError("Grounding DINO posthoc cần GPU runtime")
        model_id = run_cfg["models"]["gdino_model"]
        revision = run_cfg["models"]["gdino_revision"]
        weight_path = hf_hub_download(model_id, filename="model.safetensors", revision=revision)
        digest = hashlib.sha256()
        with open(weight_path, "rb") as fin:
            for chunk in iter(lambda: fin.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
        self.weight_sha256 = digest.hexdigest()
        expected = run_cfg["models"]["gdino_weight_sha256"]
        if self.weight_sha256 != expected:
            raise RuntimeError(f"Grounding DINO checksum mismatch: {self.weight_sha256} != {expected}")
        self.processor = AutoProcessor.from_pretrained(model_id, revision=revision)
        # Quan trọng: model, pixel_values và toàn bộ forward đều FP32. Không dùng autocast/half.
        self.model = AutoModelForZeroShotObjectDetection.from_pretrained(
            model_id, revision=revision, use_safetensors=True, dtype=torch.float32
        ).to(self.device).float().eval()
        parameter = next(self.model.parameters())
        if parameter.dtype != torch.float32:
            raise RuntimeError(f"GDINO parameter dtype phải là float32, nhận {parameter.dtype}")
        self._warmup()
        print(
            "Grounding DINO posthoc ready:", model_id, "@", revision[:12],
            "| fp32 | sha256:", self.weight_sha256[:16], flush=True,
        )

    @staticmethod
    def _match_label(raw, labels):
        text = str(raw).strip().lower().strip(" .")
        if text in labels:
            return text
        matches = [label for label in labels if label in text or text in label]
        return min(matches, key=len) if matches else None

    def _inputs(self, images, labels):
        try:
            inputs = self.processor(
                images=images, text=[labels for _ in images], padding=True, return_tensors="pt"
            )
        except Exception:
            prompt = ". ".join(labels) + "."
            inputs = self.processor(
                images=images, text=[prompt] * len(images), padding=True, return_tensors="pt"
            )
        inputs = inputs.to(self.device)
        if "pixel_values" in inputs:
            inputs["pixel_values"] = inputs["pixel_values"].float()
        if "pixel_mask" in inputs:
            inputs["pixel_mask"] = inputs["pixel_mask"].to(dtype=torch.long)
        return inputs

    @torch.inference_mode()
    def _warmup(self):
        probe = Image.new("RGB", (96, 64), (127, 127, 127))
        try:
            inputs = self._inputs([probe], ["person"])
            outputs = self.model(**inputs)
            if not torch.isfinite(outputs.logits).all():
                raise RuntimeError("warm-up logits chứa NaN/Inf")
        except Exception as exc:
            raise RuntimeError(f"Grounding DINO FP32 warm-up thất bại trước khi tạo namespace: {exc!r}") from exc
        finally:
            probe.close()

    def _postprocess(self, outputs, inputs, target_sizes):
        inf = self.cfg["inference"]
        kwargs = {
            "threshold": inf["gdino_box_threshold"],
            "text_threshold": inf["gdino_text_threshold"],
            "target_sizes": target_sizes,
        }
        try:
            return self.processor.post_process_grounded_object_detection(outputs, **kwargs)
        except TypeError:
            try:
                return self.processor.post_process_grounded_object_detection(
                    outputs, inputs.get("input_ids"), **kwargs
                )
            except TypeError:
                kwargs["box_threshold"] = kwargs.pop("threshold")
                return self.processor.post_process_grounded_object_detection(
                    outputs, inputs.get("input_ids"), **kwargs
                )

    @torch.inference_mode()
    def _predict_once(self, images, genre):
        target_sizes = [(image.height, image.width) for image in images]
        output = [[] for _ in images]
        vocab_chunk = int(self.cfg["inference"].get("gdino_vocab_chunk", 20))
        for labels in batched(_labels_for_genre(self.cfg, genre), vocab_chunk):
            inputs = self._inputs(images, labels)
            results = self._postprocess(self.model(**inputs), inputs, target_sizes)
            for image_i, result in enumerate(results):
                boxes = result["boxes"].detach().float().cpu().numpy()
                scores = result["scores"].detach().float().cpu().numpy()
                raw_labels = result.get("text_labels", result.get("labels", []))
                for box, score, raw in zip(boxes, scores, raw_labels):
                    label = self._match_label(raw, labels)
                    if label:
                        output[image_i].append({
                            "label": label,
                            "box": [float(value) for value in box],
                            "conf": float(score),
                            "sources": ["gdino"],
                            "source_scores": {"gdino": float(score)},
                        })
        return output

    def predict(self, images, genre):
        try:
            return self._predict_once(images, genre)
        except torch.cuda.OutOfMemoryError:
            if len(images) == 1:
                raise
            torch.cuda.empty_cache()
            middle = len(images) // 2
            print(f"GDINO FP32 CUDA OOM batch={len(images)}; retry {middle}+{len(images)-middle}")
            return self.predict(images[:middle], genre) + self.predict(images[middle:], genre)


def _count_decision(run_cfg, det):
    canonical = det["canonical_label"]
    if canonical not in set(run_cfg["metadata"]["countable_classes"]):
        return False, False, None, "not_countable"
    scores = det.get("source_scores") or {source: det["conf"] for source in det.get("sources", [])}
    margins = []
    for source, score in scores.items():
        source_thresholds = run_cfg["metadata"]["count_thresholds"].get(source, {})
        threshold = float(source_thresholds.get(
            canonical, source_thresholds.get("_default", run_cfg["inference"]["count_conf"])
        ))
        margins.append((float(score) - threshold, source, threshold))
    if not margins:
        threshold = float(run_cfg["inference"]["count_conf"])
        return True, float(det["conf"]) >= threshold, threshold, "global_fallback"
    margin, source, threshold = max(margins)
    return True, margin >= 0, threshold, source


def _bbox_list(det):
    box = det["bbox_px"]
    return [float(box["x1"]), float(box["y1"]), float(box["x2"]), float(box["y2"])]


def _refresh_summary(run_cfg, doc, old_uncertain_reasons):
    detections = doc["detections"]
    detections.sort(key=lambda det: (-float(det["conf"]), det["canonical_label"], det["bbox_norm"]["x1"]))
    for index, det in enumerate(detections):
        det["instance_id"] = f"{doc['video_id']}:{doc['frame_name']}:{index:03d}"

    counted = [det for det in detections if det.get("counted")]
    grouped = defaultdict(list)
    for det in counted:
        grouped[det["canonical_label"]].append(det)
    crowd_present = any(det["canonical_label"] == "crowd" for det in detections)
    uncertain = set(doc.get("count_uncertain_labels", []))
    if crowd_present:
        uncertain.add("person")
        old_uncertain_reasons.setdefault("person", set()).add("crowd_present")

    object_counts = []
    for label, items in sorted(grouped.items()):
        reasons = sorted(old_uncertain_reasons.get(label, {"source_document_uncertain"} if label in uncertain else set()))
        object_counts.append({
            "label": label,
            "count": len(items),
            "countable": True,
            "max_conf": round(max(float(item["conf"]) for item in items), 5),
            "mean_conf": round(sum(float(item["conf"]) for item in items) / len(items), 5),
            "aliases": sorted({alias for item in items for alias in item.get("aliases", [])}),
            "is_estimate": True,
            "is_lower_bound": label in uncertain,
            "lower_bound_reasons": reasons,
            "count_quality": "lower_bound" if label in uncertain else "detector_exact_candidate",
            "thresholds_used": sorted({round(float(item["count_threshold_used"]), 5) for item in items}),
            "sources": sorted({item["count_source"] for item in items}),
        })

    labels = sorted({det["label"] for det in detections})
    canonical_labels = sorted({det["canonical_label"] for det in detections})
    spatial_tokens = sorted({
        f"{det['canonical_label']}@{position}"
        for det in detections for position in det.get("position_tags", [])
    })
    count_tokens = sorted(f"{item['label']}:{item['count']}" for item in object_counts)
    phrases = [f"{item['count']} {item['label']}" for item in object_counts]
    seen_phrases = set()
    for det in counted:
        color = f"{det['dominant_color']} " if det.get("color_reliable") else ""
        phrase = f"{color}{det['canonical_label']} at {det['position']}"
        if phrase not in seen_phrases:
            phrases.append(phrase)
            seen_phrases.add(phrase)
    phrases.extend(sorted({
        term for det in detections for term in run_cfg["vi_aliases"].get(det["canonical_label"], [])
    }))

    notes = list(doc.get("count_notes", [])) or ["visible detector instances; occluded objects may be missed"]
    if crowd_present and not any("crowd detected" in note for note in notes):
        notes.append("crowd detected: person count is a lower bound, not crowd population")
    doc.update({
        "detection_count": len(detections),
        "counted_detection_count": len(counted),
        "labels": labels,
        "canonical_labels": canonical_labels,
        "spatial_tokens": spatial_tokens,
        "count_tokens": count_tokens,
        "object_text": "; ".join(phrases),
        "object_counts": object_counts,
        "crowd_present": crowd_present,
        "counts_are_estimates": True,
        "count_is_lower_bound": bool(uncertain),
        "count_uncertain_labels": sorted(uncertain),
        "count_notes": notes,
    })


def _merge_gdino(run_cfg, doc, image, gdino_detections):
    detections = [copy.deepcopy(det) for det in doc.get("detections", [])]
    old_uncertain_reasons = defaultdict(set)
    for item in doc.get("object_counts", []):
        old_uncertain_reasons[item["label"]].update(item.get("lower_bound_reasons", []))
    color = ColorExtractor(run_cfg)
    width, height = image.size
    dedup_iou = float(run_cfg["inference"]["canonical_dedup_iou"])
    min_area = float(run_cfg["inference"]["min_box_area_norm"])
    grid_size = int(run_cfg["metadata"]["grid_size"])

    for candidate in sorted(gdino_detections, key=lambda item: float(item["conf"]), reverse=True):
        canonical = _canonical(run_cfg, candidate["label"])
        match = next((
            item for item in detections
            if item["canonical_label"] == canonical and box_iou(_bbox_list(item), candidate["box"]) >= dedup_iou
        ), None)
        if match is not None:
            old_conf = float(match["conf"])
            merged_box = weighted_box(_bbox_list(match), max(0.01, old_conf), candidate["box"], max(0.01, candidate["conf"]))
            if float(candidate["conf"]) > old_conf:
                match["label"] = candidate["label"]
            match["conf"] = round(max(old_conf, float(candidate["conf"])), 5)
            match["aliases"] = sorted(set(match.get("aliases", [])) | {
                candidate["label"], canonical, *run_cfg["vi_aliases"].get(canonical, [])
            })
            match["sources"] = sorted(set(match.get("sources", [])) | {"gdino"})
            source_scores = {key: float(value) for key, value in match.get("source_scores", {}).items()}
            source_scores["gdino"] = max(source_scores.get("gdino", 0.0), float(candidate["conf"]))
            match["source_scores"] = {key: round(value, 5) for key, value in sorted(source_scores.items())}
            # Giữ metadata màu mask YOLOE; mask không được lưu để có thể tái tạo chính xác.
            match.update(position_metadata(merged_box, width, height, grid_size))
            countable, counted, threshold, count_source = _count_decision(run_cfg, match)
            match.update({
                "countable": countable, "counted": counted,
                "count_threshold_used": threshold, "count_source": count_source,
            })
            continue

        spatial = position_metadata(candidate["box"], width, height, grid_size)
        if spatial["size_norm"]["area"] < min_area:
            continue
        raw = {
            "label": candidate["label"],
            "canonical_label": canonical,
            "conf": round(float(candidate["conf"]), 5),
            "sources": ["gdino"],
            "source_scores": {"gdino": round(float(candidate["conf"]), 5)},
        }
        countable, counted, threshold, count_source = _count_decision(run_cfg, raw)
        detections.append({
            "instance_id": "",
            "label": candidate["label"],
            "canonical_label": canonical,
            "aliases": sorted({candidate["label"], canonical, *run_cfg["vi_aliases"].get(canonical, [])}),
            "label_group": run_cfg["label_groups"].get(canonical, "other"),
            "conf": raw["conf"],
            "countable": countable,
            "counted": counted,
            "count_threshold_used": threshold,
            "count_source": count_source,
            "sources": ["gdino"],
            "source_scores": raw["source_scores"],
            # GDINO không có instance mask: inner-box Lab được lưu nhưng color_reliable=False.
            **color.extract(image, candidate["box"], canonical, None, candidate["conf"]),
            **spatial,
        })

    doc["detections"] = detections
    _refresh_summary(run_cfg, doc, old_uncertain_reasons)
    return doc


def _bind_augmented_document(doc, augment_cfg, source_prefix, reasons, attempted):
    doc["pipeline_version"] = augment_cfg["pipeline_version"]
    doc["config_hash"] = augment_cfg["config_hash"]
    doc["runtime_hash"] = augment_cfg["runtime_hash"]
    doc["generated_at"] = datetime.now(timezone.utc).isoformat()
    doc["fallback_status"] = "used_posthoc_fp32" if attempted else "not_required_posthoc"
    doc["fallback_reasons"] = list(reasons)
    doc.pop("fallback_error", None)
    pipeline = dict(doc.get("pipeline", {}))
    pipeline.update({
        "runtime_hash": augment_cfg["runtime_hash"],
        "gdino_mode": "posthoc_policy_fp32",
        "gdino_model": augment_cfg["models"]["gdino_model"],
        "gdino_revision": augment_cfg["models"]["gdino_revision"],
        "gdino_weight_sha256": augment_cfg["runtime"]["fallback_weight_sha256"],
        "gdino_precision": "float32",
        "posthoc_augmentation_version": GDINO_POSTHOC_CODE_VERSION,
        "posthoc_source_prefix": source_prefix,
        "posthoc_targeted": bool(attempted),
    })
    doc["pipeline"] = pipeline
    return doc


def _download_with_retry(store, entry, attempts=6):
    for attempt in range(attempts):
        try:
            return store.download_one(entry)
        except Exception:
            if attempt + 1 == attempts:
                raise
            time.sleep(min(20.0, 1.5 * (2 ** attempt)))


def _stage_targets(store, entries):
    staged = [None] * len(entries)
    workers = min(GDINO_POSTHOC_DOWNLOAD_WORKERS, max(1, len(entries)))
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(_download_with_retry, store, entry): index for index, entry in enumerate(entries)}
        for future in as_completed(futures):
            staged[futures[future]] = future.result()
    return staged


def run_gdino_posthoc(base_cfg, stable_manifest, source_prefix_override=None):
    # Dùng current cfg nếu còn nguyên runtime cell 8. Sau Colab restart có thể restore exact source
    # config từ R2 bằng GDINO_SOURCE_PREFIX, không cần gọi run_pipeline/YOLOE.
    bootstrap_store = R2Store(base_cfg)
    bootstrap_store.check()
    if source_prefix_override:
        source_prefix = source_prefix_override.rstrip("/")
        source_cfg = json.loads(_r2_payload(bootstrap_store, f"{source_prefix}/config.json"))
        source_cfg["r2"]["output_prefix"] = source_prefix
    else:
        source_cfg = copy.deepcopy(base_cfg)
        source_prefix = source_cfg.get("r2", {}).get("output_prefix")
    if not source_prefix or not source_cfg.get("runtime_hash"):
        raise RuntimeError(
            "cfg chưa bind source runtime. Nếu Colab đã restart, gán exact Final R2 output vào "
            "GDINO_SOURCE_PREFIX; không cần chạy lại YOLOE."
        )
    if source_cfg.get("manifest_hash") != base_cfg.get("manifest_hash"):
        raise RuntimeError(
            f"Source manifest {source_cfg.get('manifest_hash')} != current manifest "
            f"{base_cfg.get('manifest_hash')}; không augment nhầm input"
        )
    if int(source_cfg.get("manifest_count", -1)) != len(stable_manifest):
        raise RuntimeError(
            f"Source manifest_count {source_cfg.get('manifest_count')} != current {len(stable_manifest)}"
        )
    store = R2Store(source_cfg)
    shard_size = int(source_cfg["run"]["shard_size"])
    total_shards = math.ceil(len(stable_manifest) / shard_size)
    source_completed = existing_shards(source_cfg, store, manifest=stable_manifest)
    missing = sorted(set(range(total_shards)) - source_completed)
    if missing:
        raise RuntimeError(
            f"YOLOE source chưa hoàn tất: thiếu {len(missing)}/{total_shards} shard, sample={missing[:20]}. "
            "Resume cell 8 trước, không augment bộ dữ liệu dở dang."
        )

    entry_by_id = {f"{entry['video_id']}:{entry['frame_name']}": entry for entry in stable_manifest}
    target_plan = {}
    target_by_reason = Counter()
    source_documents = 0
    print(f"Scan {total_shards} YOLOE shard để tính lại fallback policy...", flush=True)
    for sid in range(total_shards):
        docs = _read_shard(store, shard_key(source_cfg, sid))
        for doc in docs:
            source_documents += 1
            if doc.get("status") != "ok":
                raise RuntimeError(f"Source có document lỗi: {doc.get('document_id')} status={doc.get('status')}")
            reasons = _posthoc_reasons(source_cfg, doc)
            if reasons:
                target_plan[doc["document_id"]] = reasons
                target_by_reason.update(reasons)
        if (sid + 1) % 25 == 0 or sid + 1 == total_shards:
            print(f"  scanned {sid+1}/{total_shards} shards | targets={len(target_plan)}", flush=True)
    if source_documents != len(stable_manifest):
        raise RuntimeError(f"Source document count {source_documents} != manifest {len(stable_manifest)}")
    if not target_plan:
        print("Không có frame nào thỏa fallback policy; giữ namespace YOLOE hiện tại.")
        return None, {"status": "no_targets", "source_prefix": source_prefix}, None
    print("Posthoc targets:", len(target_plan), "| reasons:", dict(target_by_reason), flush=True)

    # Full run đã trả detector local; dọn cache trước khi load GDINO FP32.
    gc.collect()
    torch.cuda.empty_cache()
    gdino = _GroundingDINOFP32(source_cfg)

    contract = {
        "version": GDINO_POSTHOC_CODE_VERSION,
        "source_prefix": source_prefix,
        "source_config_hash": source_cfg["config_hash"],
        "source_runtime_hash": source_cfg["runtime_hash"],
        "manifest_hash": source_cfg["manifest_hash"],
        "gdino_model": source_cfg["models"]["gdino_model"],
        "gdino_revision": source_cfg["models"]["gdino_revision"],
        "gdino_weight_sha256": gdino.weight_sha256,
        "precision": "float32",
        "policy": {
            key: source_cfg["inference"].get(key)
            for key in (
                "gdino_trigger_top_conf", "gdino_min_detections_by_genre",
                "gdino_anchor_classes_by_genre", "gdino_audit_rate_by_genre",
                "gdino_vocab_chunk", "gdino_box_threshold", "gdino_text_threshold",
                "canonical_dedup_iou", "min_box_area_norm",
            )
        },
    }
    augment_cfg = copy.deepcopy(source_cfg)
    augment_cfg["pipeline_version"] = "aic26-od-v5.1-gdino-augment"
    augment_cfg["inference"]["gdino_mode"] = "posthoc_policy_fp32"
    augment_cfg["inference"]["gdino_batch_size"] = GDINO_POSTHOC_BATCH_SIZE
    augment_cfg["config_hash"] = _stable_hash(contract, 12)
    augment_runtime = copy.deepcopy(source_cfg["runtime"])
    augment_runtime.update({
        "fallback_precision": "float32",
        "augmentation_version": GDINO_POSTHOC_CODE_VERSION,
        "augmentation_source_config_hash": source_cfg["config_hash"],
        "augmentation_source_runtime_hash": source_cfg["runtime_hash"],
        "augmentation_contract_sha256": _stable_hash(contract),
        "fallback_weight_sha256": gdino.weight_sha256,
    })
    augment_cfg["runtime"] = augment_runtime
    augment_cfg["runtime_hash"] = _stable_hash(augment_runtime, 16)
    augment_cfg["r2"]["output_prefix"] = (
        f"{augment_cfg['r2']['output_root']}/{augment_cfg['pipeline_version']}/"
        f"{augment_cfg['config_hash']}/input-{augment_cfg['manifest_hash']}/"
        f"source-{source_cfg['runtime_hash']}/runtime-{augment_cfg['runtime_hash']}"
    )
    output_prefix = augment_cfg["r2"]["output_prefix"]
    print("Augmented R2 output:", output_prefix, flush=True)

    local_root = Path(augment_cfg["local"]["output"]) / augment_cfg["config_hash"] / augment_cfg["runtime_hash"]
    shard_dir = local_root / "shards"
    shard_dir.mkdir(parents=True, exist_ok=True)
    config_path = local_root / "config.json"
    config_path.write_text(json.dumps(augment_cfg, ensure_ascii=False, indent=2), encoding="utf-8")
    store.upload_file(config_path, f"{output_prefix}/config.json", content_type="application/json")
    augmented_manifest_path = local_root / "manifest.jsonl.gz"
    write_gzip_jsonl(augmented_manifest_path, stable_manifest)
    store.upload_file(
        augmented_manifest_path, f"{output_prefix}/manifest.jsonl.gz",
        content_type="application/x-ndjson", content_encoding="gzip",
    )
    plan_path = local_root / "augmentation-plan.json.gz"
    with gzip.open(plan_path, "wt", encoding="utf-8") as fout:
        json.dump({
            "contract": contract,
            "target_count": len(target_plan),
            "target_by_reason": dict(target_by_reason),
            "targets": target_plan,
        }, fout, ensure_ascii=False, separators=(",", ":"))
    store.upload_file(plan_path, f"{output_prefix}/augmentation-plan.json.gz", content_type="application/json", content_encoding="gzip")

    completed = existing_shards(augment_cfg, store, manifest=stable_manifest)
    pending = [sid for sid in range(total_shards) if sid not in completed]
    print(f"Augment shards total={total_shards} completed={len(completed)} pending={len(pending)}", flush=True)
    started = time.time()
    newly_committed = 0
    newly_targeted = 0
    gdino_raw_detections = 0

    for position, sid in enumerate(pending, 1):
        rows = stable_manifest[sid * shard_size:(sid + 1) * shard_size]
        docs = _read_shard(store, shard_key(source_cfg, sid))
        docs_by_id = {doc["document_id"]: doc for doc in docs}
        target_entries = [entry for entry in rows if f"{entry['video_id']}:{entry['frame_name']}" in target_plan]
        staged, images = [], {}
        try:
            if target_entries:
                staged = _stage_targets(store, target_entries)
                for entry in staged:
                    doc_id = f"{entry['video_id']}:{entry['frame_name']}"
                    with Image.open(entry["local_path"]) as raw:
                        images[doc_id] = raw.convert("RGB")
                by_genre = defaultdict(list)
                for entry in target_entries:
                    by_genre[entry["genre"]].append(f"{entry['video_id']}:{entry['frame_name']}")
                for genre, doc_ids in sorted(by_genre.items()):
                    for id_batch in batched(doc_ids, GDINO_POSTHOC_BATCH_SIZE):
                        results = gdino.predict([images[doc_id] for doc_id in id_batch], genre)
                        for doc_id, result in zip(id_batch, results):
                            gdino_raw_detections += len(result)
                            docs_by_id[doc_id] = _merge_gdino(
                                source_cfg, docs_by_id[doc_id], images[doc_id], result
                            )

            output_docs = []
            for entry in rows:
                doc_id = f"{entry['video_id']}:{entry['frame_name']}"
                doc = docs_by_id[doc_id]
                reasons = target_plan.get(doc_id, [])
                output_docs.append(_bind_augmented_document(
                    doc, augment_cfg, source_prefix, reasons, attempted=bool(reasons)
                ))
            if len(output_docs) != len(rows):
                raise RuntimeError(f"Augment shard {sid}: output {len(output_docs)} != input {len(rows)}")

            local = write_gzip_jsonl(shard_dir / f"part-{sid:06d}.jsonl.gz", output_docs)
            store.upload_file(
                local, shard_key(augment_cfg, sid),
                content_type="application/x-ndjson", content_encoding="gzip",
            )
            marker = {
                "shard_id": sid,
                "document_count": len(output_docs),
                "error_count": 0,
                "sha256": hashlib.sha256(local.read_bytes()).hexdigest(),
                "manifest_hash": augment_cfg["manifest_hash"],
                "runtime_hash": augment_cfg["runtime_hash"],
                "resolved_detector": augment_cfg["runtime"]["resolved_detector"],
                "weight_sha256": augment_cfg["runtime"]["weight_sha256"],
                "text_encoder_sha256": augment_cfg["runtime"]["text_encoder_sha256"],
                "prompt_embeddings_sha256": augment_cfg["runtime"]["prompt_embeddings_sha256"],
                "segmentation_dtype_guard": augment_cfg["runtime"]["segmentation_dtype_guard"],
                "fallback_weight_sha256": augment_cfg["runtime"]["fallback_weight_sha256"],
                "manifest_slice_hash": hashlib.sha256(
                    "\n".join(entry["r2_key"] for entry in rows).encode("utf-8")
                ).hexdigest(),
                "config_hash": augment_cfg["config_hash"],
                "first_document_id": output_docs[0]["document_id"],
                "last_document_id": output_docs[-1]["document_id"],
                "source_prefix": source_prefix,
                "posthoc_target_count": len(target_entries),
                "committed_at": time.time(),
            }
            marker_path = shard_dir / f"part-{sid:06d}.complete.json"
            marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8")
            # Marker luôn là commit point cuối cùng.
            store.upload_file(marker_path, completion_key(augment_cfg, sid), content_type="application/json")
            newly_committed += 1
            newly_targeted += len(target_entries)
            elapsed = max(1e-9, time.time() - started)
            print(
                f"[{position}/{len(pending)}] shard={sid:06d} frames={len(rows)} "
                f"gdino_targets={len(target_entries)} committed=True "
                f"elapsed_min={elapsed/60:.1f}", flush=True,
            )
        finally:
            for image in images.values():
                image.close()
            if staged:
                store.cleanup_entries(staged)

    final_completed = existing_shards(augment_cfg, store, manifest=stable_manifest)
    if len(final_completed) != total_shards:
        raise RuntimeError(f"Augmented output mới có {len(final_completed)}/{total_shards} committed shards")
    result = {
        "status": "complete",
        "source_prefix": source_prefix,
        "output_prefix": output_prefix,
        "planned_target_frames": len(target_plan),
        "target_by_reason": dict(target_by_reason),
        "newly_processed_target_frames": newly_targeted,
        "newly_committed_shards": newly_committed,
        "gdino_raw_detections_this_run": gdino_raw_detections,
        "completed_shards": len(final_completed),
        "elapsed_seconds_this_run": round(time.time() - started, 2),
        "config_hash": augment_cfg["config_hash"],
        "manifest_hash": augment_cfg["manifest_hash"],
        "runtime_hash": augment_cfg["runtime_hash"],
    }
    summary_path = local_root / "last_run_summary.json"
    summary_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    store.upload_file(summary_path, f"{output_prefix}/last_run_summary.json", content_type="application/json")
    return augment_cfg, result, target_plan


if RUN_GDINO_POSTHOC:
    gdino_aug_cfg, gdino_aug_summary, gdino_aug_target_plan = run_gdino_posthoc(
        cfg, manifest, source_prefix_override=GDINO_SOURCE_PREFIX
    )
    print(json.dumps(gdino_aug_summary, ensure_ascii=False, indent=2))
else:
    gdino_aug_cfg = None
    print("Bỏ qua posthoc. Đặt RUN_GDINO_POSTHOC=True để chạy sau khi YOLOE full hoàn tất.")


In [ ]:
# === 10. Validate toàn bộ output đã có trên R2 ===
from od_runner import build_contact_sheet, evaluate_ground_truth, validate_output

# Nếu cell 9 đã augment thành công thì validate namespace v5.1; nếu không thì validate YOLOE v5.
validation_cfg = gdino_aug_cfg if globals().get("gdino_aug_cfg") is not None else cfg
validation = validate_output(validation_cfg, manifest=manifest, max_shards=None, sample_size=200)
print(json.dumps(validation, ensure_ascii=False, indent=2))
assert validation["ok"], "Validation lỗi; xem validation['errors']. Shard thiếu marker sẽ tự retry ở lần run sau."

contact_sheet = build_contact_sheet(validation_cfg, manifest, validation["sample_document_ids"])
print("Contact sheet:", contact_sheet)

# Optional ground truth nhỏ (khuyến nghị >=200 frame, stratified theo genre/class).
# JSONL contract: {"document_id":"K01_V001:000123","detections":[
#   {"canonical_label":"person","bbox_norm":{"x1":0.1,"y1":0.1,"x2":0.3,"y2":0.8}}]}
GROUND_TRUTH_JSONL = None
if GROUND_TRUTH_JSONL:
    gt_metrics = evaluate_ground_truth(validation_cfg, GROUND_TRUTH_JSONL, iou_threshold=0.5, per_frame_k=5)
    print(json.dumps(gt_metrics, ensure_ascii=False, indent=2))


In [ ]:
%%writefile od_elastic.py
import gzip
import hashlib
import io
import json
import os
import re
import statistics
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

from od_r2 import R2Store, iter_gzip_jsonl_stream


DEFAULT_COUNTABLE_CLASSES = frozenset({
    "person", "car", "motorcycle", "bicycle", "bus", "truck", "van", "train",
    "airplane", "helicopter", "boat", "ambulance", "fire truck", "microphone",
    "camera", "cell phone", "laptop", "screen", "table", "chair", "backpack",
    "handbag", "bottle", "book", "document", "sign", "banner", "license plate",
    "flag", "helmet", "plate", "bowl", "pan", "pot", "knife", "fruit",
})


def index_mapping():
    return {
        "settings": {
            "number_of_shards": 3,
            "number_of_replicas": 0,
            "refresh_interval": "30s",
        },
        "mappings": {
            "dynamic": "strict",
            "properties": {
                "schema_version": {"type": "keyword"},
                "pipeline_version": {"type": "keyword"},
                "config_hash": {"type": "keyword"},
                "manifest_hash": {"type": "keyword"},
                "runtime_hash": {"type": "keyword"},
                "resolved_detector": {"type": "keyword"},
                "weight_sha256": {"type": "keyword"},
                "text_encoder_sha256": {"type": "keyword"},
                "prompt_embeddings_sha256": {"type": "keyword"},
                "document_id": {"type": "keyword"},
                "keyframe_id": {"type": "keyword"},
                "video_id": {"type": "keyword"},
                "frame_name": {"type": "keyword"},
                "frame_index": {"type": "integer"},
                "batch": {"type": "keyword"},
                "genre": {"type": "keyword"},
                "r2_key": {"type": "keyword", "index": False, "doc_values": False},
                "image_url": {"type": "keyword", "index": False, "doc_values": False},
                "input_etag": {"type": "keyword"},
                "status": {"type": "keyword"},
                "fallback_status": {"type": "keyword"},
                "fallback_reasons": {"type": "keyword"},
                "fallback_error": {"type": "text", "index": False},
                "error": {"type": "text", "index": False},
                "generated_at": {"type": "date"},
                "width": {"type": "integer"},
                "height": {"type": "integer"},
                "detection_count": {"type": "integer"},
                "counted_detection_count": {"type": "integer"},
                "labels": {"type": "keyword"},
                "canonical_labels": {"type": "keyword"},
                "spatial_tokens": {"type": "keyword"},
                "count_tokens": {"type": "keyword"},
                "object_text": {"type": "text", "analyzer": "standard"},
                "crowd_present": {"type": "boolean"},
                "counts_are_estimates": {"type": "boolean"},
                "count_is_lower_bound": {"type": "boolean"},
                "count_uncertain_labels": {"type": "keyword"},
                "count_notes": {"type": "keyword", "index": False},
                "pipeline": {"type": "object", "enabled": False},
                "object_counts": {
                    "type": "nested",
                    "dynamic": "strict",
                    "properties": {
                        "label": {"type": "keyword"},
                        "count": {"type": "integer"},
                        "countable": {"type": "boolean"},
                        "max_conf": {"type": "half_float"},
                        "mean_conf": {"type": "half_float"},
                        "aliases": {"type": "keyword"},
                        "is_estimate": {"type": "boolean"},
                        "is_lower_bound": {"type": "boolean"},
                        "lower_bound_reasons": {"type": "keyword"},
                        "count_quality": {"type": "keyword"},
                        "thresholds_used": {"type": "half_float"},
                        "sources": {"type": "keyword"},
                    },
                },
                "detections": {
                    "type": "nested",
                    "dynamic": "strict",
                    "properties": {
                        "instance_id": {"type": "keyword"},
                        "label": {"type": "keyword"},
                        "canonical_label": {"type": "keyword"},
                        "aliases": {"type": "keyword"},
                        "label_group": {"type": "keyword"},
                        "conf": {"type": "half_float"},
                        "countable": {"type": "boolean"},
                        "counted": {"type": "boolean"},
                        "count_threshold_used": {"type": "half_float"},
                        "count_source": {"type": "keyword"},
                        "sources": {"type": "keyword"},
                        "source_scores": {"type": "object", "enabled": False},
                        "dominant_color": {"type": "keyword"},
                        "color_names": {"type": "keyword"},
                        "colors": {"type": "object", "enabled": False},
                        "color_method": {"type": "keyword"},
                        "color_confidence": {"type": "half_float"},
                        "color_purity": {"type": "half_float"},
                        "color_entropy": {"type": "half_float"},
                        "color_delta_e_p50": {"type": "half_float"},
                        "color_coverage": {"type": "half_float"},
                        "color_reliable": {"type": "boolean"},
                        "bbox_norm": {
                            "properties": {
                                "x1": {"type": "float"}, "y1": {"type": "float"},
                                "x2": {"type": "float"}, "y2": {"type": "float"},
                            }
                        },
                        "bbox_px": {
                            "properties": {
                                "x1": {"type": "integer"}, "y1": {"type": "integer"},
                                "x2": {"type": "integer"}, "y2": {"type": "integer"},
                            }
                        },
                        "center_norm": {"properties": {"x": {"type": "float"}, "y": {"type": "float"}}},
                        "size_norm": {
                            "properties": {
                                "width": {"type": "float"}, "height": {"type": "float"},
                                "area": {"type": "float"},
                            }
                        },
                        "grid7": {
                            "properties": {
                                "row": {"type": "byte"}, "col": {"type": "byte"},
                                "id": {"type": "keyword"},
                            }
                        },
                        "horizontal_zone": {"type": "keyword"},
                        "vertical_zone": {"type": "keyword"},
                        "position": {"type": "keyword"},
                        "position_tags": {"type": "keyword"},
                    },
                },
            },
        },
    }


def elastic_secrets(cfg=None):
    url = os.environ.get("ELASTIC_URL", "").rstrip("/")
    api_key = os.environ.get("ELASTIC_API_KEY", "")
    alias = os.environ.get("ELASTIC_ALIAS", "aic26-keyframe-objects")
    suffix = (
        f"{cfg.get('config_hash','manual')}-{cfg.get('manifest_hash','input')}-{cfg.get('runtime_hash','runtime')}"
        if cfg else "manual"
    )
    index = os.environ.get("ELASTIC_INDEX") or f"{alias}-{suffix}"
    if not url or not api_key:
        raise RuntimeError("Thiếu ELASTIC_URL hoặc ELASTIC_API_KEY")
    return url, api_key, index, alias


def ensure_index(url, api_key, index):
    headers = {"Authorization": f"ApiKey {api_key}", "Content-Type": "application/json"}
    head = requests.head(f"{url}/{index}", headers=headers, timeout=30)
    if head.status_code == 404:
        response = requests.put(f"{url}/{index}", headers=headers, json=index_mapping(), timeout=60)
        response.raise_for_status()
        print("Created index:", index)
    else:
        head.raise_for_status()
        current = requests.get(f"{url}/{index}/_mapping", headers=headers, timeout=30)
        current.raise_for_status()
        root = next(iter(current.json().values()))
        props = root.get("mappings", {}).get("properties", {})
        if (
            props.get("detections", {}).get("type") != "nested"
            or props.get("object_counts", {}).get("type") != "nested"
            or "count_uncertain_labels" not in props
            or "runtime_hash" not in props
        ):
            raise RuntimeError(
                f"Index {index!r} đã tồn tại nhưng không có schema v5 runtime-locked. "
                "Dùng ELASTIC_INDEX mới hoặc migrate index cũ."
            )
        print("Index exists:", index)


def _encode_operations(operations):
    chunks = []
    for action, doc in operations:
        chunks.append(json.dumps(action, separators=(",", ":"), ensure_ascii=False))
        chunks.append(json.dumps(doc, separators=(",", ":"), ensure_ascii=False))
    return ("\n".join(chunks) + "\n").encode("utf-8")


def _post_bulk(url, api_key, operations, retries=6):
    headers = {"Authorization": f"ApiKey {api_key}", "Content-Type": "application/x-ndjson"}
    pending = list(operations)
    succeeded = 0
    for attempt in range(retries):
        payload = _encode_operations(pending)
        response = requests.post(f"{url}/_bulk", headers=headers, data=payload, timeout=180)
        if response.status_code in {408, 429, 502, 503, 504}:
            time.sleep(min(30, 2 ** attempt))
            continue
        response.raise_for_status()
        body = response.json()
        items = body.get("items", [])
        if len(items) != len(pending):
            raise RuntimeError(f"Bulk response item count {len(items)} != request {len(pending)}")
        retry_items, fatal = [], []
        for operation, item in zip(pending, items):
            result = next(iter(item.values()))
            status = int(result.get("status", 0))
            if 200 <= status < 300:
                succeeded += 1
            elif status in {408, 429, 502, 503, 504}:
                retry_items.append(operation)
            else:
                fatal.append({"id": result.get("_id"), "status": status, "error": result.get("error")})
        if fatal:
            raise RuntimeError(f"Elasticsearch non-retryable bulk item errors: {fatal[:10]}")
        if not retry_items:
            return succeeded
        pending = retry_items
        time.sleep(min(30, 2 ** attempt))
    raise RuntimeError(f"Elasticsearch bulk còn {len(pending)} item sau {retries} lần retry")


def switch_alias(url, api_key, alias, new_index):
    headers = {"Authorization": f"ApiKey {api_key}", "Content-Type": "application/json"}
    current = requests.get(f"{url}/_alias/{alias}", headers=headers, timeout=30)
    old_indices = [] if current.status_code == 404 else list(current.json())
    if current.status_code != 404:
        current.raise_for_status()
    actions = [{"remove": {"index": old, "alias": alias}} for old in old_indices if old != new_index]
    actions.append({"add": {"index": new_index, "alias": alias, "is_write_index": True}})
    response = requests.post(f"{url}/_aliases", headers=headers, json={"actions": actions}, timeout=60)
    response.raise_for_status()
    return {"alias": alias, "previous_indices": old_indices, "active_index": new_index}


def _index_document_ids(url, api_key, index, page_size=5000):
    headers = {"Authorization": f"ApiKey {api_key}", "Content-Type": "application/json"}
    response = requests.post(
        f"{url}/{index}/_search?scroll=2m", headers=headers,
        json={"size": page_size, "sort": ["_doc"], "_source": False, "query": {"match_all": {}}},
        timeout=120,
    )
    response.raise_for_status()
    body, ids = response.json(), []
    scroll_id = body.get("_scroll_id")
    try:
        while True:
            hits = body.get("hits", {}).get("hits", [])
            if not hits:
                break
            ids.extend(hit["_id"] for hit in hits)
            response = requests.post(
                f"{url}/_search/scroll", headers=headers,
                json={"scroll": "2m", "scroll_id": scroll_id}, timeout=120,
            )
            response.raise_for_status()
            body = response.json()
            scroll_id = body.get("_scroll_id", scroll_id)
    finally:
        if scroll_id:
            requests.delete(
                f"{url}/_search/scroll", headers=headers, json={"scroll_id": [scroll_id]}, timeout=30
            )
    return ids


def _release_report(store, cfg):
    key = f"{cfg['r2']['output_prefix']}/validation/release-report.json"
    try:
        return json.loads(store.client.get_object(Bucket=store.bucket, Key=key)["Body"].read())
    except Exception as exc:
        raise RuntimeError(f"Thiếu/không đọc được validation artifact {key}: {exc!r}") from exc


def bulk_index_from_r2(
    cfg, max_shards=None, docs_per_bulk=500, max_bulk_bytes=8 * 1024 * 1024,
    activate_alias=False,
):
    url, api_key, index, alias = elastic_secrets(cfg)
    ensure_index(url, api_key, index)
    store = R2Store(cfg)
    from od_runner import completion_key, existing_shards, shard_key
    shard_ids = sorted(existing_shards(cfg, store))
    if max_shards is not None:
        shard_ids = shard_ids[:max_shards]
    operations, r2_document_ids, marker_fingerprints = [], [], []
    buffered_bytes = indexed = expected_documents = 0
    for shard_no, sid in enumerate(shard_ids, 1):
        key = shard_key(cfg, sid)
        marker = json.loads(
            store.client.get_object(Bucket=store.bucket, Key=completion_key(cfg, sid))["Body"].read()
        )
        payload = store.client.get_object(Bucket=store.bucket, Key=key)["Body"].read()
        payload_sha = hashlib.sha256(payload).hexdigest()
        if payload_sha != marker.get("sha256"):
            raise RuntimeError(f"Shard checksum mismatch trước bulk: {sid}")
        marker_fingerprints.append(f"{sid}:{payload_sha}")
        for doc in iter_gzip_jsonl_stream(io.BytesIO(payload)):
            action = {"index": {"_index": index, "_id": doc["document_id"]}}
            operations.append((action, doc))
            r2_document_ids.append(doc["document_id"])
            expected_documents += 1
            buffered_bytes += len(json.dumps(action)) + len(json.dumps(doc, ensure_ascii=False)) + 2
            if len(operations) >= docs_per_bulk or buffered_bytes >= max_bulk_bytes:
                indexed += _post_bulk(url, api_key, operations)
                operations, buffered_bytes = [], 0
        print(f"Elastic shard {shard_no}/{len(shard_ids)} | indexed={indexed}", flush=True)
    if operations:
        indexed += _post_bulk(url, api_key, operations)
    auth = {"Authorization": f"ApiKey {api_key}"}
    requests.post(f"{url}/{index}/_refresh", headers=auth, timeout=60).raise_for_status()
    count_response = requests.get(f"{url}/{index}/_count", headers=auth, timeout=60)
    count_response.raise_for_status()
    index_count = int(count_response.json()["count"])
    if index_count != expected_documents:
        raise RuntimeError(f"Index count {index_count} != committed R2 documents {expected_documents}; không đổi alias")
    if len(r2_document_ids) != len(set(r2_document_ids)):
        raise RuntimeError("Committed R2 shards có duplicate document_id; không đổi alias")
    r2_ids_hash = hashlib.sha256("\n".join(sorted(r2_document_ids)).encode("utf-8")).hexdigest()
    alias_result = None
    if activate_alias:
        if max_shards is not None:
            raise RuntimeError("Không được activate alias từ partial max_shards import")
        report = _release_report(store, cfg)
        expected_contract = {
            "config_hash": cfg.get("config_hash"), "manifest_hash": cfg.get("manifest_hash"),
            "runtime_hash": cfg.get("runtime_hash"),
        }
        if not report.get("validation_ok") or any(report.get(k) != v for k, v in expected_contract.items()):
            raise RuntimeError(f"Validation artifact không hợp lệ/khác runtime: {expected_contract}")
        if report.get("committed_shards") != len(shard_ids) or report.get("expected_documents") != expected_documents:
            raise RuntimeError("Validation artifact shard/document count không khớp R2 committed output")
        if report.get("document_ids_sha256") != r2_ids_hash:
            raise RuntimeError("Validation artifact document ID set không khớp R2; không đổi alias")
        marker_set_hash = hashlib.sha256("\n".join(sorted(marker_fingerprints)).encode("utf-8")).hexdigest()
        if report.get("marker_set_sha256") != marker_set_hash:
            raise RuntimeError("Completion-marker/checksum set đã đổi sau validation; không đổi alias")
        index_ids = _index_document_ids(url, api_key, index)
        if len(index_ids) != len(set(index_ids)):
            raise RuntimeError("Elasticsearch index có duplicate _id bất thường")
        index_ids_hash = hashlib.sha256("\n".join(sorted(index_ids)).encode("utf-8")).hexdigest()
        if index_ids_hash != r2_ids_hash:
            raise RuntimeError("Elasticsearch _id set không khớp validation artifact; không đổi alias")
        alias_result = switch_alias(url, api_key, alias, index)
    return {
        "index": index, "alias": alias, "alias_activated": bool(alias_result),
        "documents_indexed": indexed, "index_document_count": index_count,
        "shards": len(shard_ids), "document_ids_sha256": r2_ids_hash,
        "alias_result": alias_result,
    }


def _inner_hits(name="matched_objects", size=20, scripted=True):
    return {
        "name": name, "size": size,
        "sort": ([{"_score": "desc"}, {"detections.conf": "desc"}] if scripted else [{"detections.conf": "desc"}]),
        "_source": {"includes": [
            "detections.instance_id", "detections.label", "detections.canonical_label",
            "detections.conf", "detections.bbox_norm", "detections.bbox_px",
            "detections.center_norm", "detections.size_norm", "detections.position",
            "detections.position_tags", "detections.dominant_color", "detections.color_names",
            "detections.sources", "detections.countable", "detections.counted",
            "detections.color_method", "detections.color_confidence", "detections.color_purity",
            "detections.color_entropy", "detections.color_delta_e_p50",
            "detections.color_coverage", "detections.color_reliable",
        ]},
    }


def _nested_script_query(filters, source, params, inner_name, size):
    nested = {"nested": {
        "path": "detections", "score_mode": "max",
        "query": {"script_score": {
            "query": {"bool": {"filter": filters}},
            "script": {"source": source, "params": params},
        }},
        "inner_hits": _inner_hits(inner_name),
    }}
    return {
        "size": size, "min_score": 0.001,
        "track_total_hits": False, "timeout": "3s",
        "query": {"bool": {"filter": [{"term": {"status": "ok"}}], "must": [nested]}},
        "sort": [{"_score": "desc"}],
    }


def _nested_native_query(filters, inner_name, size, prefer_color=False):
    nested_filter = {"bool": {"filter": filters}}
    sort = []
    if prefer_color:
        sort.append({"detections.color_confidence": {
            "order": "desc", "mode": "max",
            "nested": {"path": "detections", "filter": nested_filter},
        }})
    sort.append({"detections.conf": {
        "order": "desc", "mode": "max",
        "nested": {"path": "detections", "filter": nested_filter},
    }})
    return {
        "size": size, "track_total_hits": False, "timeout": "3s",
        "query": {"bool": {"filter": [
            {"term": {"status": "ok"}},
            {"nested": {
                "path": "detections", "score_mode": "none", "query": nested_filter,
                "inner_hits": _inner_hits(inner_name, scripted=False),
            }},
        ]}},
        "sort": sort,
    }


def query_object(
    label, position=None, color=None, min_conf=0.30, size=50,
    require_reliable_color=True, color_mode="dominant",
):
    filters = [
        {"term": {"detections.canonical_label": label}},
        {"range": {"detections.conf": {"gte": min_conf}}},
    ]
    if position:
        filters.append({"term": {"detections.position_tags": position}})
    if color:
        if color_mode not in {"dominant", "any"}:
            raise ValueError("color_mode phải là 'dominant' hoặc 'any'")
        color_field = "detections.dominant_color" if color_mode == "dominant" else "detections.color_names"
        filters.append({"term": {color_field: color}})
        if require_reliable_color:
            filters.append({"term": {"detections.color_reliable": True}})
    targets = {
        "top_left": (1/6, 1/6), "top_center": (0.5, 1/6), "top_right": (5/6, 1/6),
        "middle_left": (1/6, 0.5), "center": (0.5, 0.5), "middle_right": (5/6, 0.5),
        "bottom_left": (1/6, 5/6), "bottom_center": (0.5, 5/6), "bottom_right": (5/6, 5/6),
        "left": (1/6, 0.5), "right": (5/6, 0.5), "top": (0.5, 1/6),
        "bottom": (0.5, 5/6), "middle": (0.5, 0.5),
    }
    if not position:
        # Label/color-only: native nested sort, không chạy Painless trên hàng triệu candidates.
        return _nested_native_query(filters, "matched_objects", size, prefer_color=bool(color))
    else:
        tx, ty = targets.get(position, (0.5, 0.5))
        grid = re.fullmatch(r"r([0-6])c([0-6])", position)
        if grid:
            ty, tx = (int(grid.group(1)) + 0.5) / 7, (int(grid.group(2)) + 0.5) / 7
        source = """
            double conf = doc['detections.conf'].value;
            double dx = doc['detections.center_norm.x'].value - params.tx;
            double dy = doc['detections.center_norm.y'].value - params.ty;
            double proximity = 1.0 - Math.min(1.0, Math.sqrt(dx*dx + dy*dy) / 1.41421356237);
            return 0.65 * conf + 0.35 * proximity + 0.0001;
        """
    return _nested_script_query(filters, source, {"tx": tx, "ty": ty}, "matched_objects", size)


def query_drawn_region(
    label, x1, y1, x2, y2, min_conf=0.30, size=50, weights=None, center_padding=0.10,
):
    """Candidate overlap rồi rank confidence + IoU + tâm + area + aspect similarity."""
    if not (0 <= x1 < x2 <= 1 and 0 <= y1 < y2 <= 1):
        raise ValueError("Rectangle phải thỏa 0 <= x1 < x2 <= 1 và 0 <= y1 < y2 <= 1")
    if not (0 <= center_padding <= 0.5):
        raise ValueError("center_padding phải trong [0,0.5]")
    filters = [
        {"term": {"detections.canonical_label": label}},
        {"range": {"detections.conf": {"gte": min_conf}}},
        {"range": {"detections.bbox_norm.x1": {"lte": x2}}},
        {"range": {"detections.bbox_norm.x2": {"gte": x1}}},
        {"range": {"detections.bbox_norm.y1": {"lte": y2}}},
        {"range": {"detections.bbox_norm.y2": {"gte": y1}}},
        {"range": {"detections.center_norm.x": {"gte": max(0, x1-center_padding), "lte": min(1, x2+center_padding)}}},
        {"range": {"detections.center_norm.y": {"gte": max(0, y1-center_padding), "lte": min(1, y2+center_padding)}}},
    ]
    weights = weights or {"confidence": 0.25, "iou": 0.35, "position": 0.20, "area": 0.12, "aspect": 0.08}
    required = {"confidence", "iou", "position", "area", "aspect"}
    if set(weights) != required or any(float(weights[k]) < 0 for k in required) or sum(float(weights[k]) for k in required) <= 0:
        raise ValueError(f"weights phải có đúng {sorted(required)}, không âm và tổng > 0")
    total_weight = sum(float(weights[k]) for k in required)
    weights = {k: float(v) / total_weight for k, v in weights.items()}
    source = """
        double ax1=doc['detections.bbox_norm.x1'].value, ay1=doc['detections.bbox_norm.y1'].value;
        double ax2=doc['detections.bbox_norm.x2'].value, ay2=doc['detections.bbox_norm.y2'].value;
        double iw=Math.min(ax2,params.x2)-Math.max(ax1,params.x1);
        double ih=Math.min(ay2,params.y2)-Math.max(ay1,params.y1);
        if (iw <= 0 || ih <= 0) return 0.0;
        double inter=iw*ih, aa=(ax2-ax1)*(ay2-ay1), qa=(params.x2-params.x1)*(params.y2-params.y1);
        double iou=inter/Math.max(1e-9,aa+qa-inter);
        double acx=(ax1+ax2)/2.0, acy=(ay1+ay2)/2.0;
        double qcx=(params.x1+params.x2)/2.0, qcy=(params.y1+params.y2)/2.0;
        double diag=Math.max(0.05,Math.sqrt((params.x2-params.x1)*(params.x2-params.x1)+(params.y2-params.y1)*(params.y2-params.y1)));
        double proximity=1.0-Math.min(1.0,Math.sqrt((acx-qcx)*(acx-qcx)+(acy-qcy)*(acy-qcy))/diag);
        double areaSim=Math.min(aa,qa)/Math.max(1e-9,Math.max(aa,qa));
        double arA=(ax2-ax1)/Math.max(1e-9,ay2-ay1), arQ=(params.x2-params.x1)/Math.max(1e-9,params.y2-params.y1);
        double aspectSim=Math.min(arA,arQ)/Math.max(arA,arQ);
        return params.wc*doc['detections.conf'].value + params.wi*iou + params.wp*proximity + params.wa*areaSim + params.wr*aspectSim + 0.0001;
    """
    params = {"x1": x1, "y1": y1, "x2": x2, "y2": y2,
              "wc": weights["confidence"], "wi": weights["iou"], "wp": weights["position"],
              "wa": weights["area"], "wr": weights["aspect"]}
    return _nested_script_query(filters, source, params, "objects_ranked_for_drawn_region", size)


def query_near_point(label, x, y, tolerance=0.10, min_conf=0.30, size=50):
    """Tìm object có tâm gần tọa độ chuẩn hóa (x,y) mà user click/vẽ."""
    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < tolerance <= 0.5):
        raise ValueError("x/y phải trong [0,1], tolerance trong (0,0.5]")
    filters = [
        {"term": {"detections.canonical_label": label}},
        {"range": {"detections.conf": {"gte": min_conf}}},
        {"range": {"detections.center_norm.x": {"gte": max(0, x-tolerance), "lte": min(1, x+tolerance)}}},
        {"range": {"detections.center_norm.y": {"gte": max(0, y-tolerance), "lte": min(1, y+tolerance)}}},
    ]
    source = """
        double dx=doc['detections.center_norm.x'].value-params.x;
        double dy=doc['detections.center_norm.y'].value-params.y;
        double proximity=1.0-Math.min(1.0,Math.sqrt(dx*dx+dy*dy)/params.tolerance);
        return 0.35*doc['detections.conf'].value + 0.65*proximity + 0.0001;
    """
    return _nested_script_query(filters, source, {"x": x, "y": y, "tolerance": tolerance}, "objects_near_point", size)


def query_count(
    label, gte=None, lte=None, exact=None, exclude_lower_bound=True, size=50,
    countable_classes=None, zero_mode="presence", presence_threshold=0.20, strict=False,
):
    allowed = set(countable_classes or DEFAULT_COUNTABLE_CLASSES)
    if label not in allowed:
        raise ValueError(f"{label!r} không thuộc COUNTABLE_CLASSES; chỉ query presence, không query exact count")
    if not (0 <= presence_threshold <= 1):
        raise ValueError("presence_threshold phải trong [0,1]")
    if exact is not None and exact < 0:
        raise ValueError("exact phải >= 0")
    if exact == 0:
        filters = [{"term": {"status": "ok"}}]
        if zero_mode not in {"presence", "counted"}:
            raise ValueError("zero_mode phải là 'presence' hoặc 'counted'")
        if zero_mode == "presence":
            must_not = [{"nested": {"path": "detections", "query": {"bool": {"filter": [
                {"term": {"detections.canonical_label": label}},
                {"range": {"detections.conf": {"gte": presence_threshold}}},
            ]}}}}]
        else:
            must_not = [{"nested": {"path": "object_counts", "query": {"term": {"object_counts.label": label}}}}]
        if exclude_lower_bound:
            must_not.append({"term": {"count_uncertain_labels": label}})
        return {"size": size, "query": {"bool": {"filter": filters, "must_not": must_not}}}
    bounds = {"gte": exact, "lte": exact} if exact is not None else {k: v for k, v in (("gte", gte), ("lte", lte)) if v is not None}
    if not bounds:
        raise ValueError("Cần exact hoặc gte/lte")
    if strict and exact is None:
        raise ValueError("strict chỉ có nghĩa với exact count")
    nested_filters = [
        {"term": {"object_counts.label": label}},
        {"range": {"object_counts.count": bounds}},
    ]
    if exclude_lower_bound or strict:
        nested_filters.append({"term": {"object_counts.is_lower_bound": False}})
    root_filter = [
        {"term": {"status": "ok"}},
        {"nested": {"path": "object_counts", "query": {"bool": {"filter": nested_filters}}, "inner_hits": {"name": "matched_counts", "size": 20}}},
    ]
    must_not = []
    if strict:
        # Không chấp nhận detection cùng class nằm dưới count threshold nhưng trên presence threshold.
        must_not.extend([
            {"term": {"count_uncertain_labels": label}},
            {"nested": {"path": "detections", "query": {"bool": {"filter": [
                {"term": {"detections.canonical_label": label}},
                {"term": {"detections.counted": False}},
                {"range": {"detections.conf": {"gte": presence_threshold}}},
            ]}}}},
        ])
    return {"size": size, "query": {"bool": {"filter": root_filter, "must_not": must_not}}}


def _percentile(values, q):
    if not values:
        return None
    values = sorted(values)
    pos = (len(values) - 1) * q
    lo, hi = int(pos), min(len(values) - 1, int(pos) + 1)
    return values[lo] + (values[hi] - values[lo]) * (pos - lo)


def evaluate_spatial_retrieval(
    url, api_key, index_or_alias, ground_truth_jsonl, ks=(10, 50, 100),
    min_conf=0.20, max_queries=None,
):
    """Cross-frame drawing retrieval: GT box -> ES query -> rank của đúng document_id."""
    queries = []
    with open(ground_truth_jsonl, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            for det in row.get("detections", []):
                b = det["bbox_norm"]
                queries.append((
                    row["document_id"], det["canonical_label"],
                    query_drawn_region(
                        det["canonical_label"], b["x1"], b["y1"], b["x2"], b["y2"],
                        min_conf=min_conf, size=max(ks),
                    ),
                ))
    if max_queries is not None:
        queries = queries[:max_queries]
    headers = {"Authorization": f"ApiKey {api_key}", "Content-Type": "application/json"}
    ranks, latencies, failures = [], [], []
    for target_id, label, body in queries:
        started = time.perf_counter()
        response = requests.post(f"{url.rstrip('/')}/{index_or_alias}/_search", headers=headers, json=body, timeout=10)
        latency_ms = (time.perf_counter() - started) * 1000
        latencies.append(latency_ms)
        if not response.ok:
            failures.append({"document_id": target_id, "label": label, "status": response.status_code})
            ranks.append(None)
            continue
        ids = [hit["_id"] for hit in response.json().get("hits", {}).get("hits", [])]
        ranks.append(ids.index(target_id) + 1 if target_id in ids else None)
    found = [rank for rank in ranks if rank is not None]
    return {
        "queries": len(queries),
        **{f"spatial_recall_at_{k}": sum(rank is not None and rank <= k for rank in ranks) / max(1, len(ranks)) for k in ks},
        "median_rank": statistics.median(found) if found else None,
        "not_found_within_max_k": sum(rank is None for rank in ranks),
        "latency_ms": {"p50": _percentile(latencies, 0.50), "p95": _percentile(latencies, 0.95), "p99": _percentile(latencies, 0.99)},
        "failures": failures[:20],
    }


def benchmark_query_latency(url, api_key, index_or_alias, query_cases, repeats=25, concurrency=8):
    """Benchmark p50/p95/p99 với concurrency gần tải thi đấu; query_cases là dict name->body."""
    headers = {"Authorization": f"ApiKey {api_key}", "Content-Type": "application/json"}
    tasks = [(name, body) for _ in range(repeats) for name, body in query_cases.items()]

    def one(name, body):
        started = time.perf_counter()
        response = requests.post(
            f"{url.rstrip('/')}/{index_or_alias}/_search", headers=headers, json=body, timeout=10
        )
        elapsed = (time.perf_counter() - started) * 1000
        return name, elapsed, response.status_code

    samples, errors = {}, []
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = [pool.submit(one, name, body) for name, body in tasks]
        for future in as_completed(futures):
            name, elapsed, status = future.result()
            samples.setdefault(name, []).append(elapsed)
            if not (200 <= status < 300):
                errors.append({"name": name, "status": status})
    return {
        "concurrency": concurrency, "repeats": repeats,
        "latency_ms": {
            name: {"p50": _percentile(v, 0.50), "p95": _percentile(v, 0.95), "p99": _percentile(v, 0.99)}
            for name, v in samples.items()
        },
        "errors": errors[:50],
    }


In [ ]:
# === 11. Elasticsearch: secrets, mapping và optional bulk upload ===
# Bulk upload là external write có chủ đích; bật cờ sau khi validate output R2.
UPLOAD_TO_ELASTIC = False
ACTIVATE_ELASTIC_ALIAS = False  # chỉ True khi muốn UI chuyển nguyên tử sang index mới đã validate
RUN_ELASTIC_BENCHMARK = False
RUN_SPATIAL_GT_EVAL = False

elastic_url = read_secret("ELASTIC_URL", "")
elastic_api_key = read_secret("ELASTIC_API_KEY", "")
elastic_index = read_secret("ELASTIC_INDEX", "")
elastic_alias = read_secret("ELASTIC_ALIAS", "aic26-keyframe-objects")
if elastic_url:
    os.environ["ELASTIC_URL"] = elastic_url
if elastic_api_key:
    os.environ["ELASTIC_API_KEY"] = elastic_api_key
if elastic_index:
    os.environ["ELASTIC_INDEX"] = elastic_index
else:
    os.environ.pop("ELASTIC_INDEX", None)
os.environ["ELASTIC_ALIAS"] = elastic_alias

from od_elastic import (
    benchmark_query_latency, bulk_index_from_r2, elastic_secrets, evaluate_spatial_retrieval,
    query_count, query_drawn_region, query_near_point, query_object,
)

print("Ví dụ: người màu xanh góc trái")
print(json.dumps(query_object("person", position="top_left", color="blue"), ensure_ascii=False, indent=2))
print("\nVí dụ: xe giao với vùng user vẽ bên trái")
print(json.dumps(query_drawn_region("car", 0.0, 0.0, 0.35, 1.0), ensure_ascii=False, indent=2))
print("\nVí dụ: micro gần tọa độ user click (0.2, 0.25)")
print(json.dumps(query_near_point("microphone", 0.2, 0.25, tolerance=0.08), ensure_ascii=False, indent=2))
print("\nVí dụ: đúng 3 người, bỏ frame crowd/lower-bound")
print(json.dumps(query_count("person", exact=3, strict=True), ensure_ascii=False, indent=2))
print("\nVí dụ: không có presence người từ recall threshold 0.20")
print(json.dumps(query_count("person", exact=0, zero_mode="presence"), ensure_ascii=False, indent=2))

if UPLOAD_TO_ELASTIC:
    elastic_summary = bulk_index_from_r2(cfg, activate_alias=ACTIVATE_ELASTIC_ALIAS)
    print(json.dumps(elastic_summary, ensure_ascii=False, indent=2))
else:
    print("\nUPLOAD_TO_ELASTIC=False: chỉ preview query, chưa ghi Elasticsearch.")

if RUN_ELASTIC_BENCHMARK:
    bench_url, bench_key, bench_index, _ = elastic_secrets(cfg)
    cases = {
        "object_person": query_object("person"),
        "object_person_top_left": query_object("person", position="top_left"),
        "draw_person": query_drawn_region("person", 0.0, 0.0, 0.35, 0.45),
        "count_person_exact_3": query_count("person", exact=3, strict=True),
    }
    latency = benchmark_query_latency(bench_url, bench_key, bench_index, cases, repeats=25, concurrency=8)
    print(json.dumps(latency, ensure_ascii=False, indent=2))

if RUN_SPATIAL_GT_EVAL:
    assert GROUND_TRUTH_JSONL, "Cần GROUND_TRUTH_JSONL để đo cross-frame spatial retrieval"
    eval_url, eval_key, eval_index, _ = elastic_secrets(cfg)
    spatial_metrics = evaluate_spatial_retrieval(
        eval_url, eval_key, eval_index, GROUND_TRUTH_JSONL, ks=(10, 50, 100)
    )
    print(json.dumps(spatial_metrics, ensure_ascii=False, indent=2))


## Schema/query contract cho backend và frontend

- Dùng `document_id = <video_id>:<frame_name>` làm Elasticsearch `_id`, nên re-index idempotent.
- `detections` là `nested`: mọi điều kiện label/conf/color/position/bbox phải nằm trong **cùng một nested query**.
- Màu chỉ có cho `COLOR_CLASSES`; ưu tiên YOLOE segmentation mask + CIE Lab. Query màu mặc định
  yêu cầu `color_reliable=true`, nên GDINO inner-box fallback không âm thầm được coi là màu chắc chắn.
- `inner_hits` trả đúng detection đã match; frontend lấy `bbox_norm` hoặc `bbox_px` để vẽ.
- Rectangle user vẽ phải normalize về `[0,1]`. Filter đầu lấy candidate overlap; `script_score`
  xếp hạng theo confidence + IoU + khoảng cách tâm + độ giống diện tích/tỉ lệ cạnh. Box chỉ chạm
  biên có score 0 và bị loại; box phủ toàn frame không còn thắng chỉ vì giao mọi vùng.
- `object_counts` cũng là `nested`; query count theo `canonical_label`, không đếm role synonym riêng.
- Count chỉ áp dụng whitelist entity rời rạc, với threshold theo class/model. `crowd` chỉ làm count
  `person` thành lower-bound; `max_det` làm mọi countable class không chắc. `exact=0` mặc định là
  `zero_mode="presence"` (không detection từ recall threshold), khác `zero_mode="counted"`.
  `strict=True` cho exact dương loại frame còn detection ngay dưới count threshold.
- `object_text` chỉ dùng cho recall/BM25 hoặc debug. Filter chính xác phải dùng keyword/nested fields.
- Crop embedding không nhét vào document frame: vector per-instance nên ở index/collection riêng để tránh làm Elasticsearch frame index phình lớn.

## Vận hành

1. Model YOLOE và Grounding-DINO được load/checksum trước resume. Final namespace gồm
   `config_hash + manifest_hash + runtime_hash`; runtime hash khóa weight, precision, Torch/CUDA,
   package environment, code `od_pipeline_lib.py/od_runner.py/od_r2.py` và checksum prompt embedding.
   Toàn bộ genre prompt được cache khi model còn FP32, cast theo predictor rồi warm-up inference thật
   trước khi namespace được bind. Segmentation postprocess có dtype guard đã self-test để bước
   `mask coefficients @ prototypes` chạy FP32 trong khi detector vẫn FP16.
2. Shard chỉ được skip khi marker parse được, đúng runtime/model/manifest slice/document count và raw
   shard còn tồn tại. Marker hỏng bị coi là pending để tự overwrite; full checksum ở validation.
3. Sau full run, `validation["ok"]` phải true; xem contact sheet 200 ảnh và chạy GT metrics nếu có nhãn.
   Metric within-frame tên đúng là `per_frame_box_recall_at_k`; cross-frame drawing retrieval dùng
   `evaluate_spatial_retrieval()` để đo Recall@10/50/100, median rank và p50/p95/p99.
4. Alias activation bắt buộc release report đúng config/manifest/runtime, marker checksum set, exact
   document-ID hash và Elasticsearch `_id` set; chỉ `_count` bằng nhau là chưa đủ.
5. Sau bulk import lớn, có thể tăng replicas/đặt refresh interval theo cluster production.

Grounding-DINO offline dùng policy theo số detection, confidence và anchor class theo genre. Truy vấn
một class hoàn toàn mới vẫn cần một on-demand re-detection worker ở backend; notebook không thể tự
phản ứng với query phát sinh sau khi batch job đã kết thúc.
